> **Research-source reading copy.** Saved outputs and machine-specific metadata were removed for sharing. These original workflow cells require the private raw/source-evidence workspace and database, which are intentionally not bundled. For an executable portable result replay, use `python scripts/replay_saved_study.py` from the repository root. See README and docs/REPRODUCIBILITY.md.

# Historical simulation VaR: measurement and validation

This notebook asks how well a historical loss threshold describes subsequent FX valuation losses for a **fixed, hypothetical, unhedged USD1 million position**, reported in PHP. It uses the frozen BSP USD-row **PHIL. PESO EQUIVALENT** series, January 2020–September 2026.

**Main finding:** the 250-change, 99% `higher` model records **22 exceptions in 1,390 targets (1.5827%)**. Exceptions are more frequent in multi-calendar-day intervals, particularly in 2021–2022. Calculations are reproducible and exception endpoints are source-matched; forecast adequacy and operational timing remain qualified.

## Reading guide

1. **Inputs and original intervals:** establish source lineage and calendar eligibility.
2. **Prior windows and valuation:** select historical changes without using the target outcome; explain signs, units and the quantile cutoff.
3. **Backtesting and sensitivity:** compare prior estimates with subsequent losses, inspect exception timing and compare variants on identical targets.
4. **Evidence and conclusion:** verify the frozen bundle, summarize coverage and yearly patterns, and state what the model can support.

An **exception** occurs when subsequent signed loss strictly exceeds prior signed VaR. Positive loss means a loss; negative loss means a gain. VaR is a loss quantile, not a maximum-loss guarantee or a bank-approved limit.

The [methodology contract](../MODELLING_CONTRACT.md) controls the model. Calendar eligibility does not establish equal market measurement horizons or historical public availability. These are retrospective published-observation forecasts, not verified market-close one-day bank risk.

**Reproduction:** run all cells in order with **Python (traded-risk-analytics)**. The notebook starts independently of EDA memory. Database access is read-only; derived ledger/window files retain their existing identical-rerun checks. The final evidence cell verifies the existing immutable bundle instead of re-exporting this revised notebook over its frozen code snapshot. The earlier `reproduce_var_validation.py` runner belongs to that original snapshot and is not the verification command for this expanded reader edition.

## 1. Source observations and original intervals

### 1. Locate the frozen inputs

The workbook hash identifies the exact source version. The seven review row-pairs are fixed by the contract, independently of their gains or losses. They remain in the ledger; eligibility flags determine how later models may use them.

In [ ]:
# Identify the exact source files before calculating anything.
# The hash acts as a fingerprint: a different input file must not silently enter this run.

from pathlib import Path
from decimal import Decimal
from fractions import Fraction
import hashlib
import json
import math
import platform

import duckdb
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

database = project_root / "traded_risk.duckdb"
contract_path = project_root / "MODELLING_CONTRACT.md"
snapshot_sha256 = "3cf2fd962d1fbd9c4ecc59f56bb71204aa5a29d5f0ee666faa956835532c6848"
source_sheet = "peso cross daily"
snapshot_path = project_root / "data/raw/bsp" / f"dailycrossrates_{snapshot_sha256}.xlsx"
calendar_path = project_root / "data/reference/bsp_missing_calendar_20261102.csv"

frozen_review_pairs = {
    (15358, 15362), (15424, 15426), (15664, 15666),
    (16314, 16316), (16345, 16349), (16646, 16650), (16687, 16691),
}

for path in [database, contract_path, snapshot_path, calendar_path]:
    if not path.is_file():
        raise FileNotFoundError(f"Required project input not found: {path}")

assert hashlib.sha256(snapshot_path.read_bytes()).hexdigest() == snapshot_sha256
calendar_sha256 = hashlib.sha256(calendar_path.read_bytes()).hexdigest()
contract_sha256 = hashlib.sha256(contract_path.read_bytes()).hexdigest()
print("Project:", project_root)
print("Frozen source and required files located.")


### 2. Read observations and check source lineage

Read all calendar rows from staging and match each to its raw record using the source hash, sheet and row. Validate dates and recorded values before calculating changes. Missing markers remain in the calendar register, while available numeric observations form the endpoint sequence.

In [ ]:
# Match raw and staging records using source identity, not their display order.
# Keep missing-rate dates in the calendar register, but use numeric rates as return endpoints.

with duckdb.connect(str(database), read_only=True) as con:
    source_calendar = con.execute(
        """
        SELECT s.snapshot_sha256, s.source_sheet, s.source_row,
               s.source_date, s.usd_php_raw AS staging_raw_value,
               s.usd_php_rate, s.rate_status, s.calendar_status,
               s.calendar_evidence_source, s.calendar_note,
               s.event_context, s.review_status, s.publication_cause_status,
               s.evidence_sha256, s.staging_version,
               r.source_date AS raw_source_date, r.usd_php_raw AS raw_rate_text
        FROM staging.bsp_usd_php AS s
        LEFT JOIN raw.bsp_usd_php AS r
          ON s.snapshot_sha256 = r.snapshot_sha256
         AND s.source_sheet = r.source_sheet
         AND s.source_row = r.source_row
        WHERE s.snapshot_sha256 = ? AND s.source_sheet = ?
        ORDER BY s.source_date
        """,
        [snapshot_sha256, source_sheet],
    ).df()

source_calendar["source_date"] = pd.to_datetime(source_calendar["source_date"])
source_calendar["raw_source_date"] = pd.to_datetime(source_calendar["raw_source_date"])
assert len(source_calendar) == 2465
assert source_calendar["source_date"].is_unique
assert source_calendar["source_row"].is_unique
assert source_calendar["source_date"].tolist() == list(pd.date_range("2020-01-01", "2026-09-30"))
assert source_calendar["source_date"].eq(source_calendar["raw_source_date"]).all()
assert source_calendar["staging_raw_value"].eq(source_calendar["raw_rate_text"]).all()
assert source_calendar["evidence_sha256"].eq(calendar_sha256).all()

observations = source_calendar.loc[
    source_calendar["rate_status"].eq("valid")
].copy().reset_index(drop=True)
assert len(observations) == 1648
assert source_calendar["usd_php_rate"].notna().sum() == len(observations)
assert source_calendar["rate_status"].eq("source_missing").sum() == 817

for row in observations.itertuples(index=False):
    raw_decimal = Decimal(row.raw_rate_text)
    assert raw_decimal.is_finite() and raw_decimal > 0
    assert math.isfinite(row.usd_php_rate) and row.usd_php_rate > 0
    assert raw_decimal == Decimal(str(row.usd_php_rate))

observations["observation_order"] = range(1, len(observations) + 1)
print(f"Validated {len(source_calendar):,} calendar rows and {len(observations):,} numeric endpoints.")
display(observations[["observation_order", "source_date", "source_row", "raw_rate_text"]].head())


### 3. Build original intervals before applying exclusions

Pair each numeric observation with the immediately preceding numeric observation. For example, the first pair is January 2 → January 3, 2020. The first observation itself has no return.

`simple_return` stores a decimal return (0.01 means +1%); `change_percent` is its percent representation. Exact numerator/denominator columns preserve the fraction calculated from the raw decimal rate strings for later arithmetic audits. Neither representation is rounded in the ledger.

In [ ]:
# Pair observations before excluding any intervals, so an exclusion cannot create a new return.
# A stored decimal return of 0.01 means 1%; exact fractions retain the original decimal-rate arithmetic.

endpoint_names = {
    "source_date": "source_date",
    "source_row": "source_row",
    "raw_rate_text": "rate_raw",
    "usd_php_rate": "rate",
    "observation_order": "observation_order",
}

# Shift the complete numeric sequence, before any eligibility filtering.
interval_ledger = pd.DataFrame(index=observations.index)
for column, name in endpoint_names.items():
    interval_ledger[f"end_{name}"] = observations[column]
    interval_ledger[f"start_{name}"] = observations[column].shift(1)

interval_ledger = interval_ledger.iloc[1:].copy().reset_index(drop=True)
for column in ["start_source_row", "end_source_row",
               "start_observation_order", "end_observation_order"]:
    interval_ledger[column] = interval_ledger[column].astype("int64")

interval_ledger.insert(0, "interval_order", range(1, len(interval_ledger) + 1))
interval_ledger["snapshot_sha256"] = snapshot_sha256
interval_ledger["source_sheet"] = source_sheet
interval_ledger["interval_id"] = [
    f"{snapshot_sha256}:{source_sheet}:{a}->{b}"
    for a, b in zip(interval_ledger["start_source_row"], interval_ledger["end_source_row"])
]
interval_ledger["gap_calendar_days"] = (
    interval_ledger["end_source_date"] - interval_ledger["start_source_date"]
).dt.days

# Exact fractions are derived from the raw decimal strings, not display values.
exact_returns = [
    Fraction(Decimal(end)) / Fraction(Decimal(start)) - 1
    for start, end in zip(interval_ledger["start_rate_raw"], interval_ledger["end_rate_raw"])
]
interval_ledger["return_numerator"] = [str(r.numerator) for r in exact_returns]
interval_ledger["return_denominator"] = [str(r.denominator) for r in exact_returns]
interval_ledger["simple_return"] = [float(r) for r in exact_returns]
interval_ledger["change_percent"] = interval_ledger["simple_return"] * 100

display(interval_ledger[[
    "interval_order", "start_source_date", "end_source_date",
    "start_source_row", "end_source_row", "gap_calendar_days",
    "start_rate_raw", "end_rate_raw", "change_percent",
]].head().round({"change_percent": 6}))


### 4. Attach calendar evidence and eligibility

Inspect the dates strictly between each pair. Explained weekends/national nonworking days qualify for the calendar-candidate pool. An unexplained or incomplete intervening date triggers review.

The calculated review pairs must match the seven frozen contract IDs exactly. Main eligibility is a calendar flag only: warm-up requirements and target timing will be checked separately when rolling windows are built. All original rows remain available for inclusion sensitivity.

In [ ]:
# Check only the calendar dates strictly between the two published observations.
# Explained gaps permit calendar candidacy; they do not establish a uniform market-time horizon.

calendar_index = source_calendar.set_index("source_date")
explained_statuses = {"weekend", "verified_national_nonworking_day"}
calendar_annotations = []

for row in interval_ledger.itertuples(index=False):
    dates_between = pd.date_range(
        row.start_source_date, row.end_source_date, freq="D", inclusive="neither"
    )
    evidence = calendar_index.reindex(dates_between)
    explained = (
        evidence["rate_status"].eq("source_missing")
        & evidence["calendar_status"].isin(explained_statuses)
    )
    unresolved_dates = evidence.index[~explained].strftime("%Y-%m-%d").tolist()
    candidate = len(unresolved_dates) == 0
    reason = (
        "adjacent_source_dates" if len(dates_between) == 0
        else "explained_calendar_gaps" if candidate
        else "unresolved_intervening_dates"
    )
    calendar_annotations.append({
        "calendar_eligibility": "candidate" if candidate else "review",
        "eligibility_reason": reason,
        "main_calendar_eligible": candidate,
        "inclusion_calendar_eligible": True,
        "gap_group": "gap_1" if row.gap_calendar_days == 1 else "gap_gt1",
        "intervening_calendar_days": len(dates_between),
        "weekend_days_between": int(evidence["calendar_status"].eq("weekend").sum()),
        "verified_nonworking_days_between": int(evidence["calendar_status"].eq("verified_national_nonworking_day").sum()),
        "review_dates_json": json.dumps(unresolved_dates),
        "calendar_statuses_json": json.dumps(evidence["calendar_status"].fillna("missing_evidence").tolist()),
    })

interval_ledger = pd.concat(
    [interval_ledger, pd.DataFrame(calendar_annotations)], axis=1
)
interval_ledger["calendar_evidence_sha256"] = calendar_sha256
interval_ledger["availability_status"] = "unverified"
interval_ledger["economic_date_mapping_status"] = "unverified"

calculated_review_pairs = set(
    interval_ledger.loc[~interval_ledger["main_calendar_eligible"],
                        ["start_source_row", "end_source_row"]]
    .itertuples(index=False, name=None)
)
assert calculated_review_pairs == frozen_review_pairs
display(interval_ledger.groupby(["calendar_eligibility", "eligibility_reason"])
        .size().reset_index(name="intervals"))


### 5. Reconcile the ledger

Expected: **1,648 observations → 1,647 original intervals → 1,640 main calendar candidates + seven review intervals**. Original adjacency and IDs must remain intact. A source gap is not filled, and excluding an interval will not create a replacement return.

In [ ]:
# These assertions stop the analysis if counts, dates or interval identities differ from the agreed inputs.
# Keeping every original interval makes later exclusions and sensitivity comparisons auditable.

assert len(interval_ledger) == len(observations) - 1 == 1647
assert interval_ledger["interval_id"].is_unique
assert interval_ledger["main_calendar_eligible"].sum() == 1640
assert (~interval_ledger["main_calendar_eligible"]).sum() == 7
assert interval_ledger["inclusion_calendar_eligible"].all()
assert interval_ledger["end_source_date"].gt(interval_ledger["start_source_date"]).all()
assert interval_ledger["gap_calendar_days"].gt(0).all()
assert (interval_ledger["intervening_calendar_days"] == interval_ledger["gap_calendar_days"] - 1).all()
assert (interval_ledger["end_observation_order"] - interval_ledger["start_observation_order"]).eq(1).all()
assert interval_ledger["start_source_row"].tolist() == observations["source_row"].iloc[:-1].tolist()
assert interval_ledger["end_source_row"].tolist() == observations["source_row"].iloc[1:].tolist()
assert interval_ledger["start_source_date"].tolist() == observations["source_date"].iloc[:-1].tolist()
assert interval_ledger["end_source_date"].tolist() == observations["source_date"].iloc[1:].tolist()

display(pd.DataFrame({
    "metric": ["Numeric observations", "Original intervals", "Main calendar candidates", "Review intervals"],
    "count": [len(observations), len(interval_ledger),
              int(interval_ledger["main_calendar_eligible"].sum()),
              int((~interval_ledger["main_calendar_eligible"]).sum())],
}))

print("Seven review intervals remain in their original positions:")
display(interval_ledger.loc[~interval_ledger["main_calendar_eligible"], [
    "interval_order", "start_source_date", "end_source_date",
    "start_source_row", "end_source_row", "gap_calendar_days",
    "change_percent", "review_dates_json",
]].round({"change_percent": 6}))


### 6. Save a reproducible derived register

Save all 1,647 intervals, including the seven review rows, to a derived CSV. The companion report records source/evidence/contract hashes, software versions and reconciliation counts. Raw decimal strings and exact return fractions remain available for later audits; floating-point columns are convenience representations.

A repeated run must produce identical ledger bytes. If an existing v1 ledger differs, stop and investigate rather than silently replace the frozen artifact. These files are not database tables or model results.

In [ ]:
# Write derived evidence using a fixed format so identical inputs produce identical bytes.
# A changed existing ledger requires investigation and a new version, rather than silent replacement.

ledger_path = project_root / "data/processed/bsp_usd_php_interval_ledger_v1.csv"
report_path = project_root / "reports/bsp_interval_ledger_v1.json"
ledger_path.parent.mkdir(parents=True, exist_ok=True)
report_path.parent.mkdir(parents=True, exist_ok=True)

ledger_bytes = interval_ledger.to_csv(
    index=False, date_format="%Y-%m-%d", float_format="%.17g", lineterminator="\n"
).encode("utf-8")
if ledger_path.exists() and ledger_path.read_bytes() != ledger_bytes:
    raise RuntimeError("Existing v1 ledger differs. Investigate and version the change before saving.")
ledger_path.write_bytes(ledger_bytes)

ledger_report = {
    "ledger_version": "1.0",
    "snapshot_sha256": snapshot_sha256,
    "calendar_evidence_sha256": calendar_sha256,
    "contract_sha256": contract_sha256,
    "ledger_sha256": hashlib.sha256(ledger_bytes).hexdigest(),
    "numeric_observations": len(observations),
    "original_intervals": len(interval_ledger),
    "main_calendar_candidates": int(interval_ledger["main_calendar_eligible"].sum()),
    "review_intervals": int((~interval_ledger["main_calendar_eligible"]).sum()),
    "frozen_review_pairs": sorted([list(pair) for pair in frozen_review_pairs]),
    "checks": "source lineage, decimal consistency, original adjacency, IDs, evidence hash and frozen exclusions passed",
    "python_version": platform.python_version(),
    "duckdb_version": duckdb.__version__,
    "pandas_version": pd.__version__,
    "availability_status": "unverified",
    "economic_horizon": "mixed-duration published-observation intervals",
    "var_computed": False,
}
report_path.write_text(json.dumps(ledger_report, indent=2) + "\n", encoding="utf-8")
assert hashlib.sha256(ledger_path.read_bytes()).hexdigest() == ledger_report["ledger_sha256"]
print("Ledger saved:", ledger_path)
print("Audit report:", report_path)


### Input reconciliation

The ledger retains **1,647 original intervals**, including **1,640 calendar candidates and seven review intervals**. Exclusions do not create replacement returns or bridge source observations. Exact numerator/denominator values preserve the decimal-input arithmetic.

The following section selects training windows. Calendar eligibility is separate from the amount of prior history available and from unverified publication timing.

## 2. Prior-window selection

Verify the frozen ledger and contract before selecting training records. The candidate pool contains 1,640 changes, while every target retains its original interval identity.

In [ ]:
# Select the main candidate pool while retaining each interval's original return and identity.
# Resetting the display index does not change the underlying source-row references.

# Verify that the saved ledger and contract have not changed.
assert (
    hashlib.sha256(ledger_path.read_bytes()).hexdigest()
    == ledger_report["ledger_sha256"]
)

assert (
    hashlib.sha256(contract_path.read_bytes()).hexdigest()
    == contract_sha256
)

# Select main calendar candidates without recomputing returns.
# Original interval IDs and sequence numbers remain unchanged.
main_candidates = (
    interval_ledger.loc[
        interval_ledger["main_calendar_eligible"]
    ]
    .sort_values("interval_order")
    .copy()
    .reset_index(drop=True)
)

# Check candidate count, identity and chronological order.
assert len(main_candidates) == 1640
assert main_candidates["interval_id"].is_unique
assert main_candidates["end_source_date"].is_monotonic_increasing

display(
    main_candidates[
        [
            "interval_order",
            "start_source_date",
            "end_source_date",
            "gap_calendar_days",
            "change_percent",
        ]
    ].head(10).round({"change_percent": 6})
)

print("Main candidate changes:", len(main_candidates))

## 2.1 Select prior 250- and 500-change windows

Use only changes ending at or before the target starting observation. Review targets are excluded in the main run; targets with insufficient history remain explicitly unscored. Sequence alignment does not certify historical public availability.

In [ ]:
# Each target gets only its own prior eligible history; the target return must stay outside training.
# Retain excluded and warm-up targets in the schedule so missing evaluations remain visible.

def select_main_windows(ledger, window_size):
    """Select prior eligible changes while preserving original intervals."""
    ordered = ledger.sort_values("interval_order").copy()

    eligible = (
        ordered.loc[ordered["main_calendar_eligible"]]
        .reset_index(drop=True)
    )

    review_ids = set(
        ordered.loc[
            ~ordered["main_calendar_eligible"],
            "interval_id",
        ]
    )

    window_records = []

    # Start only when a full training window is available.
    for position in range(window_size, len(eligible)):
        target = eligible.iloc[position]

        # The right boundary is excluded: target is not in training.
        training = eligible.iloc[
            position - window_size : position
        ]

        # Check original sequence, endpoint timing and eligibility.
        assert len(training) == window_size

        assert training["interval_order"].lt(
            target["interval_order"]
        ).all()

        assert training["end_observation_order"].le(
            target["start_observation_order"]
        ).all()

        assert training["end_source_date"].le(
            target["start_source_date"]
        ).all()

        assert target["interval_id"] not in set(
            training["interval_id"]
        )

        assert set(training["interval_id"]).isdisjoint(review_ids)

        window_records.append({
            "window_id": (
                f"main:n{window_size}:"
                f"target:{target['interval_order']}"
            ),
            "eligibility_regime": "main_exclusion",
            "window_changes": window_size,
            "target_interval_order": int(target["interval_order"]),
            "target_interval_id": target["interval_id"],
            "target_start": target["start_source_date"],
            "target_end": target["end_source_date"],
            "target_gap_group": target["gap_group"],
            "training_start": training.iloc[0]["start_source_date"],
            "training_end": training.iloc[-1]["end_source_date"],
            "training_span_calendar_days": int(
                (
                    training.iloc[-1]["end_source_date"]
                    - training.iloc[0]["start_source_date"]
                ).days
            ),
            "training_gap_gt1_count": int(
                training["gap_calendar_days"].gt(1).sum()
            ),
            "training_interval_orders_json": json.dumps(
                training["interval_order"].astype(int).tolist()
            ),
        })

    # Count eligible changes strictly before each original target.
    candidate_flag = ordered["main_calendar_eligible"].astype(int)
    prior_count = candidate_flag.cumsum() - candidate_flag

    schedule = ordered[
        [
            "interval_order",
            "interval_id",
            "start_source_date",
            "end_source_date",
            "gap_group",
        ]
    ].copy()

    schedule["window_changes"] = window_size
    schedule["prior_main_candidate_count"] = prior_count

    schedule["target_status"] = [
        "excluded_review" if not candidate else
        "insufficient_history" if count < window_size else
        "ready"
        for candidate, count in zip(
            ordered["main_calendar_eligible"],
            prior_count,
        )
    ]

    return pd.DataFrame(window_records), schedule


# Build both window-size plans.
window_parts = []
schedule_parts = []

for window_size in [250, 500]:
    windows, schedule = select_main_windows(
        interval_ledger,
        window_size,
    )

    window_parts.append(windows)
    schedule_parts.append(schedule)

main_window_plan = pd.concat(window_parts, ignore_index=True)
main_target_schedule = pd.concat(schedule_parts, ignore_index=True)

display(
    main_target_schedule
    .groupby(["window_changes", "target_status"])
    .size()
    .reset_index(name="targets")
)

## 2.2 Reconcile target coverage

The main plans contain 1,390 ready targets at 250 changes and 1,140 at 500 changes. Their common target set contains 1,140 intervals, allowing comparisons on the same outcomes.

In [ ]:
# Reconcile both window plans before estimating risk.
# Common targets allow a fair comparison: both models must face exactly the same subsequent outcomes.

# Summarize coverage for each window size.
window_coverage = (
    main_window_plan
    .groupby("window_changes", sort=True)
    .agg(
        ready_targets=("target_interval_id", "size"),
        first_training_start=("training_start", "first"),
        first_training_end=("training_end", "first"),
        first_target_start=("target_start", "first"),
        first_target_end=("target_end", "first"),
        last_target_end=("target_end", "last"),
    )
    .reset_index()
)

# Each window must have a unique identity.
assert main_window_plan["window_id"].is_unique

# Every original interval must appear in each target schedule.
assert len(main_target_schedule) == 2 * len(interval_ledger)

# Check ready-target counts.
assert (
    window_coverage
    .set_index("window_changes")["ready_targets"]
    .to_dict()
) == {
    250: 1390,
    500: 1140,
}

# Check the first target interval for each window size.
expected_first_targets = {
    250: ("2021-01-19", "2021-01-20"),
    500: ("2022-01-20", "2022-01-21"),
}

for window_size, expected in expected_first_targets.items():
    rows = main_window_plan.loc[
        main_window_plan["window_changes"].eq(window_size)
    ]

    first_window = rows.iloc[0]

    actual_dates = tuple(
        str(first_window[column].date())
        for column in ["target_start", "target_end"]
    )

    assert actual_dates == expected

    # Reconcile the schedule with the actual window plan.
    status = main_target_schedule.loc[
        main_target_schedule["window_changes"].eq(window_size)
    ]

    assert status["target_status"].eq("excluded_review").sum() == 7

    assert (
        status["target_status"].eq("insufficient_history").sum()
        == window_size
    )

    ready_ids = set(
        status.loc[
            status["target_status"].eq("ready"),
            "interval_id",
        ]
    )

    assert ready_ids == set(rows["target_interval_id"])

# Identify targets evaluable under each window size.
ids_250 = set(
    main_window_plan.loc[
        main_window_plan["window_changes"].eq(250),
        "target_interval_id",
    ]
)

ids_500 = set(
    main_window_plan.loc[
        main_window_plan["window_changes"].eq(500),
        "target_interval_id",
    ]
)

# Every 500-window target should also be evaluable under 250.
assert ids_500.issubset(ids_250)

# Intersection: targets available under BOTH window sizes.
common_main_target_ids = ids_250 & ids_500

assert len(common_main_target_ids) == 1140

main_window_plan["in_common_250_500_targets"] = (
    main_window_plan["target_interval_id"].isin(
        common_main_target_ids
    )
)

display(window_coverage)

print(
    "Common main targets for fair window-size comparison:",
    len(common_main_target_ids),
)

## 2.3 Inspect the first training window

The first 250-change forecast covers January 19–20, 2021. Its training changes span January 2, 2020–January 19, 2021: 250 changes are not 250 calendar days.

In [ ]:
# Inspect one complete training membership before applying the process to every target.
# Original ledger keys let the displayed rows be traced back to both source endpoints.

# Select the first ready target with a 250-change training window.
example_window = main_window_plan.loc[
    main_window_plan["window_changes"].eq(250)
].iloc[0]

# Read its saved list of original interval-order keys.
training_keys = json.loads(
    example_window["training_interval_orders_json"]
)

# Resolve those keys back to the original ledger rows.
example_training = (
    interval_ledger
    .set_index("interval_order", drop=False)
    .loc[training_keys]
)

# Confirm complete, correctly ordered, eligible membership.
assert len(example_training) == 250
assert example_training["interval_order"].tolist() == training_keys
assert example_training["main_calendar_eligible"].all()

# Show training coverage separately from the target interval.
display(
    pd.DataFrame([
        example_window[
            [
                "window_changes",
                "target_start",
                "target_end",
                "training_start",
                "training_end",
                "training_span_calendar_days",
            ]
        ]
    ])
)

# Inspect the beginning and end of the training window.
display(
    pd.concat([
        example_training.head(3),
        example_training.tail(3),
    ])[
        [
            "interval_order",
            "start_source_date",
            "end_source_date",
            "start_source_row",
            "end_source_row",
            "change_percent",
        ]
    ].round({"change_percent": 6})
)

## 2.4 Save window memberships

Save original interval memberships and explicit target statuses with their source hashes. Memberships make it possible to trace which historical changes supported each estimate.

In [ ]:
# Save the selection plan separately from the risk estimates.
# This records which historical changes were available under the sequence rule, not verified release times.

plan_path = (
    project_root
    / "data/processed/bsp_main_rolling_windows_v1.csv"
)

schedule_path = (
    project_root
    / "data/processed/bsp_main_target_schedule_v1.csv"
)

window_report_path = (
    project_root
    / "reports/bsp_main_rolling_windows_v1.json"
)


def save_identical_or_new_csv(frame, path):
    """Save a new CSV or reproduce an identical existing version."""
    content = frame.to_csv(
        index=False,
        date_format="%Y-%m-%d",
        lineterminator="\n",
    ).encode("utf-8")

    if path.exists() and path.read_bytes() != content:
        raise RuntimeError(
            "Existing v1 artifact differs; investigate before "
            f"replacement: {path}"
        )

    path.write_bytes(content)
    return hashlib.sha256(content).hexdigest()


# Check both destinations before writing either CSV.
for frame, path in [
    (main_window_plan, plan_path),
    (main_target_schedule, schedule_path),
]:
    content = frame.to_csv(
        index=False,
        date_format="%Y-%m-%d",
        lineterminator="\n",
    ).encode("utf-8")

    if path.exists() and path.read_bytes() != content:
        raise RuntimeError(
            "Existing v1 artifact differs; investigate before "
            f"replacement: {path}"
        )


# Save the two CSVs and record their fingerprints.
plan_sha256 = save_identical_or_new_csv(
    main_window_plan,
    plan_path,
)

schedule_sha256 = save_identical_or_new_csv(
    main_target_schedule,
    schedule_path,
)

window_report = {
    "version": "1.0",
    "eligibility_regime": "main_exclusion",
    "ledger_sha256": ledger_report["ledger_sha256"],
    "contract_sha256": contract_sha256,
    "plan_sha256": plan_sha256,
    "schedule_sha256": schedule_sha256,
    "ready_targets_250": len(ids_250),
    "ready_targets_500": len(ids_500),
    "common_main_targets": len(common_main_target_ids),
    "review_targets_excluded_per_window_size": 7,
    "membership_key": (
        "training_interval_orders_json -> interval_order "
        "in the hash-identified original ledger -> interval_id"
    ),
    "checks": (
        "full window size, prior recorded sequence and dates, "
        "no review IDs, target exclusion, schedule reconciliation "
        "and common-target counts passed"
    ),
    "availability_status": "unverified",
    "var_computed": False,
}

window_report_path.write_text(
    json.dumps(window_report, indent=2) + "\n",
    encoding="utf-8",
)

print("Window plan saved:", plan_path)
print("Target schedule saved:", schedule_path)
print("Audit report:", window_report_path)

## 3. Translate historical returns into peso losses

Apply each historical percentage change to the target starting position value. At the first target, USD1 million × PHP48.066/USD = PHP48.066 million. A negative FX return produces a positive loss for this long-USD position.

In [ ]:
# Revalue historical percentage moves at the current target's starting peso value.
# For a long USD position, a falling PHP-per-USD rate creates a loss; a rising rate creates a gain.

usd_position = 1_000_000

# Locate the target associated with the first 250-change window.
target = (
    interval_ledger
    .set_index("interval_order", drop=False)
    .loc[int(example_window["target_interval_order"])]
)

# Use only the target's STARTING rate for scenario valuation.
target_start_rate_exact = Fraction(
    Decimal(target["start_rate_raw"])
)

starting_value_exact = usd_position * target_start_rate_exact

# Verify training membership and recorded chronology.
assert len(example_training) == 250
assert example_training["interval_order"].tolist() == json.loads(
    example_window["training_interval_orders_json"]
)
assert target["interval_id"] not in set(
    example_training["interval_id"]
)
assert example_training["end_source_date"].le(
    target["start_source_date"]
).all()

# Keep historical scenario references separate from target dates.
scenario_table = example_training[
    [
        "interval_order",
        "interval_id",
        "start_source_date",
        "end_source_date",
        "change_percent",
        "return_numerator",
        "return_denominator",
    ]
].copy().rename(columns={
    "interval_order": "historical_interval_order",
    "interval_id": "historical_interval_id",
    "start_source_date": "historical_start",
    "end_source_date": "historical_end",
    "change_percent": "historical_change_percent",
})

# Recover exact decimal simple returns, not rounded percentages.
historical_returns_exact = [
    Fraction(int(numerator), int(denominator))
    for numerator, denominator in zip(
        scenario_table["return_numerator"],
        scenario_table["return_denominator"],
    )
]

# Positive loss = loss; negative loss = gain.
scenario_losses_exact = [
    -starting_value_exact * historical_return
    for historical_return in historical_returns_exact
]

scenario_table["scenario_loss_numerator"] = [
    str(loss.numerator) for loss in scenario_losses_exact
]
scenario_table["scenario_loss_denominator"] = [
    str(loss.denominator) for loss in scenario_losses_exact
]
scenario_table["scenario_loss_php"] = [
    float(loss) for loss in scenario_losses_exact
]
scenario_table["scenario_pnl_php"] = (
    -scenario_table["scenario_loss_php"]
)

display(pd.DataFrame({
    "target_start": [target["start_source_date"]],
    "target_end": [target["end_source_date"]],
    "usd_position": [usd_position],
    "starting_rate_php_per_usd": [float(target_start_rate_exact)],
    "starting_position_value_php": [float(starting_value_exact)],
    "historical_scenarios": [len(scenario_table)],
}))

display(
    scenario_table[
        [
            "historical_interval_order",
            "historical_start",
            "historical_end",
            "historical_change_percent",
            "scenario_pnl_php",
            "scenario_loss_php",
        ]
    ].head(10).round({
        "historical_change_percent": 6,
        "scenario_pnl_php": 2,
        "scenario_loss_php": 2,
    })
)

## 3.1 Check signs and units

A 1% decline gives a loss equal to 1% of starting peso value; no change gives zero; a 1% increase gives a gain. Check every scenario before estimating a quantile.

In [ ]:
# Use simple known movements to check the direction and units of the scenario calculation.
# Positive loss must equal negative P&L; zero movement must produce zero loss.

# Check down 1%, unchanged, up 1%, then all historical scenarios.
test_returns = [
    Fraction(-1, 100),
    Fraction(0),
    Fraction(1, 100),
]

test_losses = [
    -starting_value_exact * r
    for r in test_returns
]

assert len(historical_returns_exact) == len(scenario_losses_exact) == 250

for r, loss in zip(
    test_returns + historical_returns_exact,
    test_losses + scenario_losses_exact,
):
    # Independent route: revalue the same USD position.
    hypothetical_rate = target_start_rate_exact * (1 + r)
    hypothetical_value = usd_position * hypothetical_rate
    direct_loss = starting_value_exact - hypothetical_value

    assert loss == direct_loss
    assert (loss > 0) == (r < 0)
    assert (loss < 0) == (r > 0)
    assert (loss == 0) == (r == 0)

print("Passed: signs, peso valuation, and all 250 scenario calculations.")

## 3.2 Select the 99% higher cutoff

Sort 250 scenario losses from smallest to largest. The `higher` rule selects rank 248, the third-largest loss. Exact values determine the cutoff; displayed rounding does not affect it.

In [ ]:
# Sort signed scenario losses, then apply the declared higher quantile rule.
# Rank 248 is the third-largest of 250 losses; it is not a maximum-loss guarantee.

import numpy as np

confidence = Fraction(99, 100)
n_scenarios = len(scenario_losses_exact)

assert n_scenarios == 250

# Sort using exact losses; preserve the matching historical references.
sorted_positions = sorted(
    range(n_scenarios),
    key=lambda i: scenario_losses_exact[i],
)

sorted_losses_exact = [
    scenario_losses_exact[i]
    for i in sorted_positions
]

ranked_scenarios = (
    scenario_table.iloc[sorted_positions]
    .copy()
    .reset_index(drop=True)
)

ranked_scenarios.insert(
    0,
    "ascending_loss_rank",
    range(1, n_scenarios + 1),
)

# Quantile position is zero-based.
# "higher" selects the next integer position when between observations.
quantile_position = (n_scenarios - 1) * confidence
cutoff_index = math.ceil(quantile_position)
cutoff_rank = cutoff_index + 1

assert cutoff_rank == 248

var99_exact = sorted_losses_exact[cutoff_index]
var99_php = float(var99_exact)

# Verify agreement with NumPy's declared quantile convention.
numpy_var99 = float(
    np.quantile(
        scenario_table["scenario_loss_php"].to_numpy(),
        0.99,
        method="higher",
    )
)

assert numpy_var99 == var99_php

ranked_scenarios["is_cutoff_rank"] = (
    ranked_scenarios["ascending_loss_rank"].eq(cutoff_rank)
)

display(pd.DataFrame({
    "target_start": [target["start_source_date"]],
    "target_end": [target["end_source_date"]],
    "scenarios": [n_scenarios],
    "confidence_pct": [99],
    "method": ["higher"],
    "cutoff_rank": [cutoff_rank],
    "var99_php": [var99_php],
    "historical_losses_above_cutoff": [
        sum(loss > var99_exact for loss in sorted_losses_exact)
    ],
    "historical_losses_equal_cutoff": [
        sum(loss == var99_exact for loss in sorted_losses_exact)
    ],
}).round({"var99_php": 2}))

# Inspect the cutoff and its neighbouring tail scenarios.
display(
    ranked_scenarios[
        [
            "ascending_loss_rank",
            "historical_interval_order",
            "historical_start",
            "historical_end",
            "scenario_loss_php",
            "is_cutoff_rank",
        ]
    ].tail(5).round({"scenario_loss_php": 2})
)

## 3.3 Compare the first forecast with its outcome

The first target gains PHP3,000, so signed loss is -PHP3,000. It does not exceed prior VaR of approximately PHP251,521.78 and is a non-exception. Under signed VaR, exception status always depends on the comparison, not the gain/loss label alone.

In [ ]:
# Keep the forecast fixed before reading the target ending rate.
# An exception requires exact signed loss to exceed exact signed VaR; display rounding cannot decide it.

# Preserve the previously calculated estimate.
prior_var99_exact = var99_exact

assert usd_position == 1_000_000
assert target["interval_id"] == example_window["target_interval_id"]

# Read the target outcome separately from historical training scenarios.
actual_start_rate = Fraction(Decimal(target["start_rate_raw"]))
actual_end_rate = Fraction(Decimal(target["end_rate_raw"]))

assert usd_position * actual_start_rate == starting_value_exact

# Positive loss = loss; negative loss = gain.
target_loss_exact = (
    usd_position * (actual_start_rate - actual_end_rate)
)

target_pnl_exact = -target_loss_exact

# Compare exact values before any display rounding.
loss_minus_var_exact = target_loss_exact - prior_var99_exact

is_exception = loss_minus_var_exact > 0
is_mathematical_tie = loss_minus_var_exact == 0

# Review flag only: this does not override strict exceedance.
near_tie_flag = (
    abs(loss_minus_var_exact) <= Fraction(1, 100)
)

if is_exception:
    classification = "exception"
elif is_mathematical_tie:
    classification = "mathematical_tie"
else:
    classification = "non_exception"

first_target_result = pd.DataFrame({
    "target_start": [target["start_source_date"]],
    "target_end": [target["end_source_date"]],
    "starting_rate": [float(actual_start_rate)],
    "ending_rate": [float(actual_end_rate)],
    "prior_var99_php": [float(prior_var99_exact)],
    "target_pnl_php": [float(target_pnl_exact)],
    "target_loss_php": [float(target_loss_exact)],
    "loss_minus_var_php": [float(loss_minus_var_exact)],
    "classification": [classification],
    "near_tie_flag": [near_tie_flag],
})

display(first_target_result.round({
    "starting_rate": 4,
    "ending_rate": 4,
    "prior_var99_php": 2,
    "target_pnl_php": 2,
    "target_loss_php": 2,
    "loss_minus_var_php": 2,
}))

print("Classification uses exact arithmetic, not rounded display values.")

## 3.4 Reusable exact-arithmetic functions

Separate estimation from target-outcome evaluation. The estimation function receives historical returns and the starting rate, not the target ending rate. Exact fractions preserve signed quantiles and strict exceedance.

In [ ]:
# Separate estimation from outcome classification so the estimate cannot use the target ending rate.
# Fraction arithmetic avoids floating-point rounding changing a tie into an exception.

def calculate_var99_exact(
    historical_returns,
    starting_rate_raw,
    method="higher",
):
    """Estimate 99% VaR without receiving the target's ending rate."""
    if method not in {"higher", "linear"}:
        raise ValueError("Supported methods: higher, linear.")

    assert len(historical_returns) in {250, 500}
    assert all(isinstance(r, Fraction) for r in historical_returns)

    starting_rate = Fraction(Decimal(starting_rate_raw))
    assert starting_rate > 0

    starting_value = 1_000_000 * starting_rate

    losses = sorted(
        -starting_value * r
        for r in historical_returns
    )

    position = (len(losses) - 1) * Fraction(99, 100)

    if method == "higher":
        return losses[math.ceil(position)]

    # Exact linear interpolation between neighbouring losses.
    lower = math.floor(position)
    upper = math.ceil(position)
    weight = position - lower

    return (
        (1 - weight) * losses[lower]
        + weight * losses[upper]
    )


def classify_loss_exact(loss, prior_var):
    """Apply strict exceedance; near-ties are separate review flags."""
    difference = loss - prior_var

    if difference > 0:
        classification = "exception"
    elif difference == 0:
        classification = "mathematical_tie"
    else:
        classification = "non_exception"

    return {
        "loss_minus_var_exact": difference,
        "classification": classification,
        "is_exception": difference > 0,
        "is_mathematical_tie": difference == 0,
        "near_tie_flag": abs(difference) <= Fraction(1, 100),
    }


def evaluate_target_var99(ledger, window_record, method="higher"):
    """Resolve saved membership, estimate prior VaR, then score outcome."""
    assert ledger["interval_order"].is_unique

    lookup = ledger.set_index("interval_order", drop=False)

    target_row = lookup.loc[
        int(window_record["target_interval_order"])
    ]

    training_keys = json.loads(
        window_record["training_interval_orders_json"]
    )

    window_size = int(window_record["window_changes"])

    assert window_size in {250, 500}
    assert len(training_keys) == window_size
    assert len(set(training_keys)) == window_size
    assert training_keys == sorted(training_keys)

    training = lookup.loc[training_keys]

    # Main-regime eligibility and original chronology checks.
    assert target_row["interval_id"] == window_record["target_interval_id"]
    assert bool(target_row["main_calendar_eligible"])
    assert training["main_calendar_eligible"].all()

    assert training["interval_order"].lt(
        target_row["interval_order"]
    ).all()

    assert training["end_observation_order"].le(
        target_row["start_observation_order"]
    ).all()

    assert training["end_source_date"].le(
        target_row["start_source_date"]
    ).all()

    historical_returns = [
        Fraction(int(numerator), int(denominator))
        for numerator, denominator in zip(
            training["return_numerator"],
            training["return_denominator"],
        )
    ]

    # Estimate first: this function receives no target ending rate.
    prior_var = calculate_var99_exact(
        historical_returns,
        target_row["start_rate_raw"],
        method=method,
    )

    # Only after estimation, read the subsequent outcome.
    start_rate = Fraction(Decimal(target_row["start_rate_raw"]))
    end_rate = Fraction(Decimal(target_row["end_rate_raw"]))

    target_loss = 1_000_000 * (start_rate - end_rate)

    return {
        "target_interval_id": target_row["interval_id"],
        "target_start": target_row["start_source_date"],
        "target_end": target_row["end_source_date"],
        "window_changes": window_size,
        "method": method,
        "var_exact": prior_var,
        "target_loss_exact": target_loss,
        "target_pnl_exact": -target_loss,
        **classify_loss_exact(target_loss, prior_var),
    }

## 3.5 Known-answer and isolation checks

Check quantile conventions, positive and negative loss signs, exact ties, near-tie review flags and first-target reproduction. These checks support numerical implementation integrity; they do not establish statistical adequacy.

In [ ]:
# Known-answer cases test the meaning of the formulas, including signed values and exact ties.
# Outcome-isolation checks verify that changing the future outcome cannot change the prior estimate.

# 1. Known-answer checks for valuation and quantile conventions.
# At PHP50/USD, USD1m is PHP50m.
base_value = Fraction(50_000_000)

# Construct returns that produce scenario losses exactly 1...250/500 pesos.
returns_250 = [
    -Fraction(i) / base_value
    for i in range(1, 251)
]

returns_500 = [
    -Fraction(i) / base_value
    for i in range(1, 501)
]

assert calculate_var99_exact(returns_250, "50", "higher") == 248
assert calculate_var99_exact(returns_250, "50", "linear") == Fraction(24751, 100)

assert calculate_var99_exact(returns_500, "50", "higher") == 496
assert calculate_var99_exact(returns_500, "50", "linear") == Fraction(49501, 100)

# 2. Falling, unchanged and rising rates; preserve negative VaR.
assert calculate_var99_exact(
    [Fraction(-1, 100)] * 250, "50"
) == 500_000

assert calculate_var99_exact(
    [Fraction(0)] * 250, "50"
) == 0

assert calculate_var99_exact(
    [Fraction(1, 100)] * 250, "50"
) == -500_000

# 3. Exact ties and genuine small exceedances.
tie = classify_loss_exact(Fraction(100), Fraction(100))
assert tie["is_mathematical_tie"]
assert not tie["is_exception"]

tiny_exceedance = classify_loss_exact(
    Fraction(100) + Fraction(1, 200),  # PHP0.005 above VaR
    Fraction(100),
)
assert tiny_exceedance["is_exception"]
assert tiny_exceedance["near_tie_flag"]

# A gain can exceed a negative signed VaR.
assert classify_loss_exact(
    Fraction(-5_000), Fraction(-10_000)
)["is_exception"]

# 4. Reproduce the manually checked first target.
first_function_result = evaluate_target_var99(
    interval_ledger,
    example_window,
    method="higher",
)

assert first_function_result["var_exact"] == prior_var99_exact
assert first_function_result["target_loss_exact"] == target_loss_exact
assert first_function_result["target_loss_exact"] == -3_000
assert first_function_result["classification"] == "non_exception"

assert round(
    float(first_function_result["var_exact"]), 2
) == 251_521.78

# 5. Perturb target/future returns and the target outcome on a COPY.
# The first target's prior VaR must remain unchanged.
perturbed = interval_ledger.copy()

target_order = int(example_window["target_interval_order"])
future_mask = perturbed["interval_order"].ge(target_order)

perturbed.loc[future_mask, "return_numerator"] = "1"
perturbed.loc[future_mask, "return_denominator"] = "1"

target_mask = perturbed["interval_order"].eq(target_order)
start_text = perturbed.loc[target_mask, "start_rate_raw"].iloc[0]

perturbed.loc[target_mask, "end_rate_raw"] = str(
    Decimal(start_text) * Decimal("1.02")
)

perturbed_result = evaluate_target_var99(
    perturbed,
    example_window,
    method="higher",
)

assert (
    perturbed_result["var_exact"]
    == first_function_result["var_exact"]
)
assert (
    perturbed_result["target_loss_exact"]
    != first_function_result["target_loss_exact"]
)

# Display the reproduced result; rounding is presentation only.
display(pd.DataFrame({
    "target_start": [first_function_result["target_start"]],
    "target_end": [first_function_result["target_end"]],
    "prior_var99_php": [
        float(first_function_result["var_exact"])
    ],
    "target_loss_php": [
        float(first_function_result["target_loss_exact"])
    ],
    "classification": [
        first_function_result["classification"]
    ],
}).round({
    "prior_var99_php": 2,
    "target_loss_php": 2,
}))

print("Passed: quantiles, signs, ties, first-target reproduction, and outcome isolation.")

## 4. Main model: 250 changes with higher

Evaluate all 1,390 ready targets using the agreed specification. The result is 22 exceptions (1.5827%), with no mathematical ties or near-tie flags. The largest loss is PHP878,000 and its exceedance above prior VaR is approximately PHP452,280.03.

In [ ]:
# Apply the same checked functions to every ready target in the main plan.
# The denominator includes all evaluated targets, not only losses or exceptions.

# Confirm that the saved ledger and window plan remain unchanged.
assert hashlib.sha256(ledger_path.read_bytes()).hexdigest() == (
    ledger_report["ledger_sha256"]
)
assert hashlib.sha256(plan_path.read_bytes()).hexdigest() == (
    window_report["plan_sha256"]
)

# Select the main 250-change windows only.
main_windows_250 = (
    main_window_plan.loc[
        main_window_plan["window_changes"].eq(250)
    ]
    .sort_values("target_interval_order")
    .copy()
)

assert len(main_windows_250) == 1390
assert main_windows_250["target_interval_id"].is_unique
assert main_windows_250["eligibility_regime"].eq(
    "main_exclusion"
).all()

# Match the plan to the ready-target schedule.
ready_250_ids = set(
    main_target_schedule.loc[
        main_target_schedule["window_changes"].eq(250)
        & main_target_schedule["target_status"].eq("ready"),
        "interval_id",
    ]
)

assert set(
    main_windows_250["target_interval_id"]
) == ready_250_ids

# Apply the previously checked function to every ready target.
main_records = []

for _, window_record in main_windows_250.iterrows():
    result = evaluate_target_var99(
        interval_ledger,
        window_record,
        method="higher",
    )

    result.update({
        "window_id": window_record["window_id"],
        "target_interval_order": int(
            window_record["target_interval_order"]
        ),
        "gap_group": window_record["target_gap_group"],
        "in_common_250_500_targets": bool(
            window_record["in_common_250_500_targets"]
        ),
    })

    main_records.append(result)

main_results_250_higher = pd.DataFrame(main_records)

# Keep exact results; add numeric columns for display and analysis.
display_fields = {
    "var_exact": "prior_var99_php",
    "target_loss_exact": "target_loss_php",
    "target_pnl_exact": "target_pnl_php",
    "loss_minus_var_exact": "loss_minus_var_php",
}

for exact_column, numeric_column in display_fields.items():
    main_results_250_higher[numeric_column] = (
        main_results_250_higher[exact_column].map(float)
    )

# Reconcile output identities and the manually checked first result.
assert len(main_results_250_higher) == 1390
assert main_results_250_higher["target_interval_id"].is_unique

assert set(
    main_results_250_higher["target_interval_id"]
) == ready_250_ids

assert (
    main_results_250_higher["in_common_250_500_targets"].sum()
    == 1140
)

first_result = main_results_250_higher.iloc[0]

assert first_result["var_exact"] == first_function_result["var_exact"]
assert (
    first_result["target_loss_exact"]
    == first_function_result["target_loss_exact"]
)
assert first_result["classification"] == "non_exception"

# Summarize the full main-run sample, retaining ties in the denominator.
evaluated_targets = len(main_results_250_higher)
exception_count = int(
    main_results_250_higher["is_exception"].sum()
)
tie_count = int(
    main_results_250_higher["is_mathematical_tie"].sum()
)

main_run_summary = pd.DataFrame({
    "model": ["250 + higher"],
    "first_target_start": [
        main_results_250_higher["target_start"].min()
    ],
    "last_target_end": [
        main_results_250_higher["target_end"].max()
    ],
    "evaluated_targets": [evaluated_targets],
    "exceptions": [exception_count],
    "exception_rate_pct": [
        exception_count / evaluated_targets * 100
    ],
    "mathematical_ties": [tie_count],
    "near_tie_flags": [
        int(main_results_250_higher["near_tie_flag"].sum())
    ],
})

display(
    main_run_summary.round({"exception_rate_pct": 4})
)

# Classification counts must sum to all evaluated targets.
classification_counts = (
    main_results_250_higher
    .groupby("classification")
    .size()
    .reset_index(name="targets")
)

assert classification_counts["targets"].sum() == 1390
display(classification_counts)

# Inspect up to ten exceptions with the largest signed exceedances.
exceptions_250 = (
    main_results_250_higher.loc[
        main_results_250_higher["is_exception"]
    ]
    .sort_values("loss_minus_var_exact", ascending=False)
)

display(
    exceptions_250[
        [
            "target_start",
            "target_end",
            "gap_group",
            "prior_var99_php",
            "target_loss_php",
            "loss_minus_var_php",
        ]
    ].head(10).round({
        "prior_var99_php": 2,
        "target_loss_php": 2,
        "loss_minus_var_php": 2,
    })
)

print("Main 250 + higher run complete: 1,390 targets evaluated.")

## 4.1 Exception timeline

Compare subsequent signed losses with estimates formed from prior changes. Red points mark exceptions. VaR adjusts as the rolling sample and starting peso value change; jumps are not proof of advance warning or immediate adaptation.

In [ ]:
# Plot prior estimates against subsequent signed losses on the original time sequence.
# Keep excluded intervals visible: joining across them would hide breaks in evaluation.

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Keep original intervals so excluded targets remain visible as gaps.
timeline = interval_ledger[
    [
        "interval_id",
        "interval_order",
        "start_source_date",
        "end_source_date",
    ]
].merge(
    main_results_250_higher[
        [
            "target_interval_id",
            "prior_var99_php",
            "target_loss_php",
            "classification",
        ]
    ],
    left_on="interval_id",
    right_on="target_interval_id",
    how="left",
    validate="one_to_one",
)

# Limit the chart to the main model's evaluation period.
first_end = main_results_250_higher["target_end"].min()
last_end = main_results_250_higher["target_end"].max()

timeline = (
    timeline.loc[
        timeline["end_source_date"].between(first_end, last_end)
    ]
    .sort_values("interval_order")
    .copy()
)

exceptions = timeline.loc[
    timeline["classification"].eq("exception")
]

excluded = timeline.loc[
    timeline["classification"].isna()
]

# Confirm that every evaluated result and exception is represented.
assert timeline["classification"].notna().sum() == 1390

assert len(exceptions) == int(
    main_results_250_higher["is_exception"].sum()
)

fig, ax = plt.subplots(figsize=(15, 6))

# Each thin stem represents one subsequent interval loss or gain.
ax.vlines(
    timeline["end_source_date"],
    0,
    timeline["target_loss_php"] / 1_000,
    color="grey",
    linewidth=0.7,
    alpha=0.45,
    label="Subsequent signed loss",
)

# Missing results remain NaN, breaking the VaR line at exclusions.
ax.plot(
    timeline["end_source_date"],
    timeline["prior_var99_php"] / 1_000,
    color="steelblue",
    linewidth=1.5,
    label="Prior 99% VaR — 250/higher",
)

ax.scatter(
    exceptions["end_source_date"],
    exceptions["target_loss_php"] / 1_000,
    color="crimson",
    edgecolors="white",
    linewidths=0.4,
    s=45,
    zorder=5,
    label=f"Exceptions ({len(exceptions)})",
)

# Mark excluded original intervals without inventing loss values.
for i, date in enumerate(excluded["end_source_date"]):
    ax.axvline(
        date,
        color="grey",
        linestyle=":",
        linewidth=0.8,
        alpha=0.5,
        label="Excluded review interval" if i == 0 else None,
    )

ax.axhline(0, color="black", linewidth=0.8)

ax.set_title(
    "Exception Timeline | Fixed USD1 Million | Main 250 + Higher"
)
ax.set_xlabel("Target ending BSP source date")
ax.set_ylabel("PHP thousands — positive = loss; negative = gain")

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
ax.legend(loc="best")
ax.margins(x=0.01)

plt.tight_layout()
plt.show()

print(
    f"{timeline['classification'].notna().sum():,} evaluated targets; "
    f"{len(exceptions)} exceptions."
)

## 4.2 Adjacent exceptions

Keep excluded targets as breaks. Among 1,385 valid adjacent pairs, no exception is immediately followed by another exception. Only 22 valid pairs follow an exception, so zero consecutive exceptions does not establish independence.

In [ ]:
# An adjacent pair is usable only when both original neighboring intervals were evaluated.
# An excluded target breaks the sequence; the next evaluated row is not automatically its neighbor.

# Keep every original interval, including skipped targets.
exception_sequence = (
    interval_ledger[
        [
            "interval_order",
            "interval_id",
            "start_source_date",
            "end_source_date",
        ]
    ]
    .merge(
        main_results_250_higher[
            [
                "target_interval_id",
                "classification",
                "is_exception",
            ]
        ],
        left_on="interval_id",
        right_on="target_interval_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values("interval_order")
    .reset_index(drop=True)
)

exception_sequence["is_evaluated"] = (
    exception_sequence["classification"].notna()
)

# Nullable Boolean preserves missing outcomes as unknown, not False.
exception_sequence["is_exception"] = (
    exception_sequence["is_exception"].astype("boolean")
)

# Shift BEFORE filtering, preserving original adjacency.
exception_sequence["previous_is_evaluated"] = (
    exception_sequence["is_evaluated"].shift(1, fill_value=False)
)
exception_sequence["previous_is_exception"] = (
    exception_sequence["is_exception"].shift(1)
)
exception_sequence["previous_interval_order"] = (
    exception_sequence["interval_order"].shift(1)
)
exception_sequence["previous_target_start"] = (
    exception_sequence["start_source_date"].shift(1)
)
exception_sequence["previous_target_end"] = (
    exception_sequence["end_source_date"].shift(1)
)

valid_pair = (
    exception_sequence["is_evaluated"]
    & exception_sequence["previous_is_evaluated"]
    & (
        exception_sequence["interval_order"]
        - exception_sequence["previous_interval_order"]
    ).eq(1)
)

adjacent_exception_pairs = (
    exception_sequence.loc[valid_pair].copy()
)

assert adjacent_exception_pairs["previous_is_exception"].notna().all()
assert adjacent_exception_pairs["is_exception"].notna().all()

assert adjacent_exception_pairs["previous_target_end"].eq(
    adjacent_exception_pairs["start_source_date"]
).all()

# Count all four transitions, including combinations with zero observations.
transition_counts = pd.crosstab(
    adjacent_exception_pairs["previous_is_exception"],
    adjacent_exception_pairs["is_exception"],
).reindex(
    index=[False, True],
    columns=[False, True],
    fill_value=0,
)

assert transition_counts.to_numpy().sum() == len(
    adjacent_exception_pairs
)

transition_table = transition_counts.rename(
    index={False: "Previous non-exception", True: "Previous exception"},
    columns={False: "Current non-exception", True: "Current exception"},
)

display(transition_table)

# Conditional exception rates, with explicit denominators.
rate_rows = []

for previous_exception in [False, True]:
    denominator = int(
        transition_counts.loc[previous_exception].sum()
    )
    exceptions_after = int(
        transition_counts.loc[previous_exception, True]
    )

    rate_rows.append({
        "previous_outcome": (
            "Exception" if previous_exception else "Non-exception"
        ),
        "valid_pairs": denominator,
        "current_exceptions": exceptions_after,
        "exception_rate_pct": (
            exceptions_after / denominator * 100
            if denominator > 0
            else float("nan")
        ),
    })

conditional_exception_rates = pd.DataFrame(rate_rows)

display(
    conditional_exception_rates.round({"exception_rate_pct": 4})
)

print(
    "Valid adjacent evaluated pairs:",
    len(adjacent_exception_pairs),
)

# Show any directly consecutive exception pairs.
consecutive_exceptions = adjacent_exception_pairs.loc[
    adjacent_exception_pairs["previous_is_exception"].eq(True)
    & adjacent_exception_pairs["is_exception"].eq(True)
]

display(
    consecutive_exceptions[
        [
            "previous_target_start",
            "previous_target_end",
            "start_source_date",
            "end_source_date",
        ]
    ]
)

## 4.3 Exception spacing

Measure separation in original interval steps and calendar days, retaining skipped intervals. Three exceptions ending July 27, August 1 and August 8, 2022 occupy a shortest inclusive span of nine original intervals. Closeness alone does not establish unusual clustering.

In [ ]:
# Measure spacing in original interval steps as well as calendar days.
# Count skipped intervals explicitly so two distant observations are not presented as consecutive trials.

# Attach the original target status for the main 250-change run.
schedule_250 = main_target_schedule.loc[
    main_target_schedule["window_changes"].eq(250),
    ["interval_id", "target_status"],
]

spacing_sequence = (
    exception_sequence[
        [
            "interval_order",
            "interval_id",
            "start_source_date",
            "end_source_date",
            "is_evaluated",
            "is_exception",
        ]
    ]
    .merge(
        schedule_250,
        on="interval_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values("interval_order")
    .reset_index(drop=True)
)

assert spacing_sequence["target_status"].notna().all()
assert spacing_sequence["is_evaluated"].eq(
    spacing_sequence["target_status"].eq("ready")
).all()

# Keep exceptions in their original chronological order.
exception_rows = spacing_sequence.loc[
    spacing_sequence["is_evaluated"]
    & spacing_sequence["is_exception"].eq(True).fillna(False)
].reset_index(drop=True)

assert len(exception_rows) >= 2, (
    "At least two exceptions are needed to calculate spacing."
)

spacing_records = []

for i in range(1, len(exception_rows)):
    previous = exception_rows.iloc[i - 1]
    current = exception_rows.iloc[i]

    # Strictly between the two exception intervals.
    between = spacing_sequence.loc[
        spacing_sequence["interval_order"].gt(
            previous["interval_order"]
        )
        & spacing_sequence["interval_order"].lt(
            current["interval_order"]
        )
    ]

    non_exception_count = int(
        (
            between["is_evaluated"]
            & between["is_exception"].eq(False).fillna(False)
        ).sum()
    )

    skipped_count = int(
        (~between["is_evaluated"]).sum()
    )

    # These are successive exceptions, so none can lie between them.
    assert not between["is_exception"].eq(True).fillna(False).any()

    original_steps = int(
        current["interval_order"] - previous["interval_order"]
    )

    assert len(between) == original_steps - 1
    assert non_exception_count + skipped_count == len(between)

    spacing_records.append({
        "previous_exception_start": previous["start_source_date"],
        "previous_exception_end": previous["end_source_date"],
        "current_exception_start": current["start_source_date"],
        "current_exception_end": current["end_source_date"],
        "calendar_days_between_ends": int(
            (
                current["end_source_date"]
                - previous["end_source_date"]
            ).days
        ),
        "original_interval_steps": original_steps,
        "intervening_non_exceptions": non_exception_count,
        "skipped_intervals_between": skipped_count,
        "crosses_skipped_interval": skipped_count > 0,
    })

exception_spacing = pd.DataFrame(spacing_records)

assert len(exception_spacing) == len(exception_rows) - 1

# Show the closest pairs first, retaining any skipped-interval flag.
exception_spacing = (
    exception_spacing
    .sort_values(
        ["original_interval_steps", "calendar_days_between_ends"]
    )
    .reset_index(drop=True)
)

display(exception_spacing)

print(
    f"{len(exception_rows)} exceptions produce "
    f"{len(exception_spacing)} successive-exception gaps."
)

# Show exactly which skipped intervals lie inside the exception date range.
first_exception_order = exception_rows.iloc[0]["interval_order"]
last_exception_order = exception_rows.iloc[-1]["interval_order"]

skipped_between_exceptions = spacing_sequence.loc[
    spacing_sequence["interval_order"].gt(first_exception_order)
    & spacing_sequence["interval_order"].lt(last_exception_order)
    & ~spacing_sequence["is_evaluated"]
]

print("Skipped intervals within the exception-to-exception coverage:")
display(
    skipped_between_exceptions[
        [
            "interval_order",
            "start_source_date",
            "end_source_date",
            "target_status",
        ]
    ]
)

## 4.4 Exploratory random-placement diagnostic

Randomly place 22 exception labels among the 1,390 evaluated positions while preserving skipped boundaries. In 20,000 placements, 11.615% have a within-block three-exception span as short or shorter than observed. This particular exploratory benchmark does not provide strong evidence of unusual closeness; it does not prove independence or rule out other dependence patterns.

In [ ]:
# This simulation asks whether a close triple can arise under a fixed-count random-placement benchmark.
# Keep the evaluation mask and skip boundaries fixed; the frequency is not a probability of independence.

import numpy as np
import matplotlib.pyplot as plt

# Preserve the complete original sequence and its evaluation mask.
placement_sequence = (
    spacing_sequence[
        [
            "interval_order",
            "end_source_date",
            "is_evaluated",
            "is_exception",
        ]
    ]
    .sort_values("interval_order")
    .reset_index(drop=True)
    .copy()
)

assert placement_sequence["interval_order"].is_unique
assert (
    placement_sequence["interval_order"].diff().dropna().eq(1).all()
)

# Each skipped interval breaks the uninterrupted evaluation block.
placement_sequence["evaluation_block"] = (
    ~placement_sequence["is_evaluated"]
).cumsum()

eligible_positions = placement_sequence.loc[
    placement_sequence["is_evaluated"]
].copy()

positions = eligible_positions["interval_order"].to_numpy(dtype=int)
blocks = eligible_positions["evaluation_block"].to_numpy(dtype=int)

observed_flags = (
    eligible_positions["is_exception"]
    .fillna(False)
    .to_numpy(dtype=bool)
)

n_targets = len(positions)
n_exceptions = int(observed_flags.sum())

assert n_targets == 1390
assert n_exceptions == 22


def shortest_triple_span(exception_positions, exception_blocks):
    """Minimum original-interval span for three exceptions in one block."""
    if len(exception_positions) < 3:
        return float("inf")

    # Positions must be chronological.
    assert np.all(np.diff(exception_positions) > 0)

    # Three successive exception labels: first through third, inclusive.
    spans = exception_positions[2:] - exception_positions[:-2] + 1

    # Matching blocks ensure no skipped interval is crossed.
    same_block = exception_blocks[2:] == exception_blocks[:-2]
    valid_spans = spans[same_block]

    return (
        int(valid_spans.min())
        if len(valid_spans)
        else float("inf")
    )


# Small known-answer checks.
assert shortest_triple_span(
    np.array([1, 4, 9]), np.array([0, 0, 0])
) == 9

assert np.isinf(shortest_triple_span(
    np.array([1, 2, 4]), np.array([0, 0, 1])
))

# Calculate the statistic from the observed exception sequence.
observed_span = shortest_triple_span(
    positions[observed_flags],
    blocks[observed_flags],
)

assert observed_span == 9

# Fixed settings: do not change them to obtain a preferred result.
simulations = 20_000
seed = 20261103
rng = np.random.default_rng(seed)

simulated_spans = np.empty(simulations)

for simulation in range(simulations):
    # Choose exactly 22 distinct evaluated targets.
    selected = np.sort(
        rng.choice(
            n_targets,
            size=n_exceptions,
            replace=False,
        )
    )

    simulated_spans[simulation] = shortest_triple_span(
        positions[selected],
        blocks[selected],
    )

# Smaller span means a more compact group.
as_close_count = int(
    (simulated_spans <= observed_span).sum()
)

random_frequency_pct = as_close_count / simulations * 100

diagnostic_summary = pd.DataFrame({
    "evaluated_targets": [n_targets],
    "exceptions_per_placement": [n_exceptions],
    "observed_shortest_span": [observed_span],
    "random_placements": [simulations],
    "placements_as_close_or_closer": [as_close_count],
    "random_frequency_pct": [random_frequency_pct],
    "seed": [seed],
})

display(
    diagnostic_summary.round({"random_frequency_pct": 4})
)

# Plot only finite spans; all simulations remain in the summary denominator.
finite_spans = simulated_spans[np.isfinite(simulated_spans)]

if len(finite_spans):
    fig, ax = plt.subplots(figsize=(11, 5))

    ax.hist(
        finite_spans,
        bins=30,
        color="steelblue",
        edgecolor="white",
    )

    ax.axvline(
        observed_span,
        color="crimson",
        linestyle="--",
        linewidth=2,
        label=f"Observed span: {observed_span} intervals",
    )

    ax.set_title("Random Placement of 22 Exceptions")
    ax.set_xlabel(
        "Shortest span containing 3 exceptions "
        "(original intervals; smaller = closer)"
    )
    ax.set_ylabel("Number of random placements")
    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.show()

print(
    "Random placements without a valid within-block triple:",
    int((~np.isfinite(simulated_spans)).sum()),
)

## 4.5 Separate calendar-gap performance

One-calendar-day targets have 12/1,069 exceptions (1.1225%); multi-calendar-day targets have 10/321 (3.1153%). These are different source-date durations. The observed difference motivates qualified reporting rather than a claim that gaps cause losses.

In [ ]:
# Report each calendar-gap group's own exception count and denominator.
# Mean exceedance describes exceptions only, whereas the exception rate uses every evaluated target.

results = main_results_250_higher.copy()

assert len(results) == 1390
assert results["target_interval_id"].is_unique
assert results["gap_group"].isin(["gap_1", "gap_gt1"]).all()

group_records = []

for gap_group in ["gap_1", "gap_gt1"]:
    group = results.loc[
        results["gap_group"].eq(gap_group)
    ]

    exceptions = group.loc[group["is_exception"]]

    n_targets = len(group)
    n_exceptions = len(exceptions)

    # Exceedance magnitudes use exact differences from the model.
    exceedances = exceptions["loss_minus_var_exact"].tolist()

    assert all(value > 0 for value in exceedances)

    group_records.append({
        "gap_group": gap_group,
        "evaluated_targets": n_targets,
        "exceptions": n_exceptions,
        "exception_rate_pct": (
            n_exceptions / n_targets * 100
            if n_targets
            else float("nan")
        ),
        "mathematical_ties": int(
            group["is_mathematical_tie"].sum()
        ),
        "mean_exceedance_php": (
            float(sum(exceedances, Fraction(0)) / n_exceptions)
            if n_exceptions
            else float("nan")
        ),
        "largest_exceedance_php": (
            float(max(exceedances))
            if n_exceptions
            else float("nan")
        ),
        "largest_target_loss_php": (
            float(group["target_loss_exact"].max())
            if n_targets
            else float("nan")
        ),
    })

gap_performance = pd.DataFrame(group_records)

# Both groups must reconcile to the full main run.
assert gap_performance["evaluated_targets"].sum() == len(results)

assert gap_performance["exceptions"].sum() == int(
    results["is_exception"].sum()
)

assert gap_performance["mathematical_ties"].sum() == int(
    results["is_mathematical_tie"].sum()
)

display(
    gap_performance.round({
        "exception_rate_pct": 4,
        "mean_exceedance_php": 2,
        "largest_exceedance_php": 2,
        "largest_target_loss_php": 2,
    })
)

## 5. Sensitivity comparisons on common targets

Restrict the main model to the same 1,140 targets available to the longer window. Its exception count becomes 15 because the evaluation sample changes. This is the comparison baseline, not a replacement for the full 1,390-target result.

In [ ]:
# Restrict the baseline to the common evaluation set before comparing configurations.
# A changed exception count here reflects a changed sample, not a changed 250/higher forecast rule.

# Select the targets available under BOTH window sizes.
common_results_250_higher = (
    main_results_250_higher.loc[
        main_results_250_higher["target_interval_id"].isin(
            common_main_target_ids
        )
    ]
    .sort_values("target_interval_order")
    .copy()
)

# Verify exact membership, not just the total count.
assert len(common_main_target_ids) == 1140
assert len(common_results_250_higher) == 1140
assert common_results_250_higher["target_interval_id"].is_unique

assert set(
    common_results_250_higher["target_interval_id"]
) == common_main_target_ids

# Confirm that filtering did not change any calculated values.
original_lookup = main_results_250_higher.set_index(
    "target_interval_id"
)
common_lookup = common_results_250_higher.set_index(
    "target_interval_id"
)

for column in [
    "var_exact",
    "target_loss_exact",
    "classification",
]:
    assert common_lookup[column].equals(
        original_lookup.loc[common_lookup.index, column]
    )


def summarize_var_results(frame):
    """Summarize frequency and severity without changing classifications."""
    n_targets = len(frame)
    exceptions = frame.loc[frame["is_exception"]]
    exceedances = exceptions["loss_minus_var_exact"].tolist()

    return {
        "evaluated_targets": n_targets,
        "exceptions": len(exceptions),
        "exception_rate_pct": (
            len(exceptions) / n_targets * 100
            if n_targets else float("nan")
        ),
        "mathematical_ties": int(
            frame["is_mathematical_tie"].sum()
        ),
        "near_tie_flags": int(
            frame["near_tie_flag"].sum()
        ),
        "mean_exceedance_php": (
            float(sum(exceedances, Fraction(0)) / len(exceedances))
            if exceedances else float("nan")
        ),
        "largest_exceedance_php": (
            float(max(exceedances))
            if exceedances else float("nan")
        ),
        "largest_target_loss_php": (
            float(frame["target_loss_exact"].max())
            if n_targets else float("nan")
        ),
    }


# Overall baseline for the common-target sample.
common_overall_250 = pd.DataFrame([{
    "model": "250 + higher",
    "sample": "Common 250/500 targets",
    "first_target_start": (
        common_results_250_higher["target_start"].min()
    ),
    "last_target_end": (
        common_results_250_higher["target_end"].max()
    ),
    **summarize_var_results(common_results_250_higher),
}])

# Gap-group baseline using the SAME common-target sample.
common_gap_250 = pd.DataFrame([
    {
        "gap_group": gap_group,
        **summarize_var_results(
            common_results_250_higher.loc[
                common_results_250_higher["gap_group"].eq(gap_group)
            ]
        ),
    }
    for gap_group in ["gap_1", "gap_gt1"]
])

# Reconcile subgroup counts to the overall common-target summary.
for column in [
    "evaluated_targets",
    "exceptions",
    "mathematical_ties",
    "near_tie_flags",
]:
    assert (
        common_gap_250[column].sum()
        == common_overall_250.iloc[0][column]
    )

display_rounding = {
    "exception_rate_pct": 4,
    "mean_exceedance_php": 2,
    "largest_exceedance_php": 2,
    "largest_target_loss_php": 2,
}

display(common_overall_250.round(display_rounding))
display(common_gap_250.round(display_rounding))

print(
    "Earlier targets outside the common sample:",
    len(main_results_250_higher) - len(common_results_250_higher),
)

## 5.1 Evaluate the 500-change alternative

Use the same targets, position and confidence level; change the training window only. A larger history is not automatically more conservative or more responsive.

In [ ]:
# Change only the history length while holding the evaluated targets and quantile convention fixed.
# More historical changes do not necessarily produce a higher loss threshold.

windows_500 = (
    main_window_plan.loc[
        main_window_plan["window_changes"].eq(500)
        & main_window_plan["target_interval_id"].isin(
            common_main_target_ids
        )
    ]
    .sort_values("target_interval_order")
    .copy()
)

assert len(windows_500) == 1140
assert windows_500["target_interval_id"].is_unique
assert set(windows_500["target_interval_id"]) == common_main_target_ids

records_500 = []

for _, window_record in windows_500.iterrows():
    result = evaluate_target_var99(
        interval_ledger,
        window_record,
        method="higher",
    )

    result.update({
        "window_id": window_record["window_id"],
        "target_interval_order": int(
            window_record["target_interval_order"]
        ),
        "gap_group": window_record["target_gap_group"],
        "in_common_250_500_targets": True,
    })

    records_500.append(result)

common_results_500_higher = pd.DataFrame(records_500)

# Preserve exact values and add numeric convenience columns.
for exact_column, numeric_column in {
    "var_exact": "prior_var99_php",
    "target_loss_exact": "target_loss_php",
    "target_pnl_exact": "target_pnl_php",
    "loss_minus_var_exact": "loss_minus_var_php",
}.items():
    common_results_500_higher[numeric_column] = (
        common_results_500_higher[exact_column].map(float)
    )

assert len(common_results_500_higher) == 1140

print("500 + higher complete: 1,140 common targets evaluated.")

## 5.2 Compare overall and gap-group results

On common targets, 250/higher has 15 exceptions versus 18 for 500/higher. Both have six multi-calendar-day exceptions, although the dates need not be identical. Counts alone do not identify a universally better model.

In [ ]:
# Compare overall and gap-specific summaries without mixing their denominators.
# Equal group exception counts can still refer to different target dates.

models_to_compare = {
    "250 + higher": common_results_250_higher,
    "500 + higher": common_results_500_higher,
}

comparison_overall = pd.DataFrame([
    {
        "model": model,
        "sample": "Same 1,140 common targets",
        **summarize_var_results(frame),
    }
    for model, frame in models_to_compare.items()
])

comparison_gap_groups = pd.DataFrame([
    {
        "model": model,
        "gap_group": gap_group,
        **summarize_var_results(
            frame.loc[frame["gap_group"].eq(gap_group)]
        ),
    }
    for model, frame in models_to_compare.items()
    for gap_group in ["gap_1", "gap_gt1"]
])

# Reconcile subgroup totals for each model.
for model in models_to_compare:
    overall = comparison_overall.loc[
        comparison_overall["model"].eq(model)
    ].iloc[0]

    groups = comparison_gap_groups.loc[
        comparison_gap_groups["model"].eq(model)
    ]

    for column in [
        "evaluated_targets",
        "exceptions",
        "mathematical_ties",
        "near_tie_flags",
    ]:
        assert groups[column].sum() == overall[column]

display(comparison_overall.round(display_rounding))
display(comparison_gap_groups.round(display_rounding))

## 5.3 Inspect paired estimate and classification changes

The 500-change estimate is higher on 379 targets, lower on 620 and equal on 141. Thirteen targets are exceptions under both models; two under 250 only and five under 500 only. These paired comparisons show sensitivity to sample composition.

In [ ]:
# Match forecasts by stable target ID and confirm their outcomes agree before measuring differences.
# Classification switches show which losses fall between the two prior estimates.

pair_columns = [
    "target_interval_id",
    "target_start",
    "target_end",
    "gap_group",
    "var_exact",
    "target_loss_exact",
    "is_exception",
]

paired_250_500 = common_results_250_higher[
    pair_columns
].merge(
    common_results_500_higher[pair_columns],
    on="target_interval_id",
    suffixes=("_250", "_500"),
    how="inner",
    validate="one_to_one",
)

assert len(paired_250_500) == 1140

# Confirm that both models scored identical target outcomes and groups.
for column in [
    "target_start",
    "target_end",
    "gap_group",
    "target_loss_exact",
]:
    assert paired_250_500[f"{column}_250"].eq(
        paired_250_500[f"{column}_500"]
    ).all()

# Signed difference: positive means the 500-window VaR is higher.
var_differences_exact = [
    var500 - var250
    for var250, var500 in zip(
        paired_250_500["var_exact_250"],
        paired_250_500["var_exact_500"],
    )
]

var_estimate_comparison = pd.DataFrame({
    "paired_targets": [len(paired_250_500)],
    "500_var_higher": [
        sum(d > 0 for d in var_differences_exact)
    ],
    "500_var_lower": [
        sum(d < 0 for d in var_differences_exact)
    ],
    "same_var": [
        sum(d == 0 for d in var_differences_exact)
    ],
    "mean_500_minus_250_php": [
        float(
            sum(var_differences_exact, Fraction(0))
            / len(var_differences_exact)
        )
    ],
})

paired_250_500["exception_pattern"] = [
    "Both exceptions" if e250 and e500 else
    "250 only" if e250 else
    "500 only" if e500 else
    "Neither"
    for e250, e500 in zip(
        paired_250_500["is_exception_250"],
        paired_250_500["is_exception_500"],
    )
]

exception_switches = (
    paired_250_500["exception_pattern"]
    .value_counts()
    .reindex(
        ["Neither", "Both exceptions", "250 only", "500 only"],
        fill_value=0,
    )
    .rename_axis("exception_pattern")
    .reset_index(name="targets")
)

assert exception_switches["targets"].sum() == 1140

display(
    var_estimate_comparison.round({
        "mean_500_minus_250_php": 2
    })
)
display(exception_switches)

## 5.4 Evaluate linear interpolation

Hold the 250-change histories and common targets fixed; change only the quantile rule. Linear interpolation combines neighboring losses rather than selecting the higher observed loss.

In [ ]:
# Hold histories and targets fixed; change only how the 99% cutoff is extracted.
# Linear interpolation combines neighboring ordered losses rather than selecting the higher observed loss.

windows_250_common = (
    main_window_plan.loc[
        main_window_plan["window_changes"].eq(250)
        & main_window_plan["target_interval_id"].isin(
            common_main_target_ids
        )
    ]
    .sort_values("target_interval_order")
    .copy()
)

assert len(windows_250_common) == 1140
assert windows_250_common["target_interval_id"].is_unique
assert set(
    windows_250_common["target_interval_id"]
) == common_main_target_ids

linear_records = []

for _, window_record in windows_250_common.iterrows():
    result = evaluate_target_var99(
        interval_ledger,
        window_record,
        method="linear",
    )

    result.update({
        "window_id": window_record["window_id"],
        "target_interval_order": int(
            window_record["target_interval_order"]
        ),
        "gap_group": window_record["target_gap_group"],
        "in_common_250_500_targets": True,
    })

    linear_records.append(result)

common_results_250_linear = pd.DataFrame(linear_records)

for exact_column, numeric_column in {
    "var_exact": "prior_var99_php",
    "target_loss_exact": "target_loss_php",
    "target_pnl_exact": "target_pnl_php",
    "loss_minus_var_exact": "loss_minus_var_php",
}.items():
    common_results_250_linear[numeric_column] = (
        common_results_250_linear[exact_column].map(float)
    )

assert len(common_results_250_linear) == 1140

print("250 + linear complete: 1,140 common targets evaluated.")

## 5.5 Compare quantile-rule summaries

The linear rule produces 17 exceptions versus 15 under higher on common targets. Both retain six multi-calendar-day exceptions. A lower estimate changes the point at which an outcome is classified as an exception.

In [ ]:
# Separate quantile-rule effects from window-length effects.
# A lower threshold can add exceptions without changing any actual target loss.

quantile_models = {
    "250 + higher": common_results_250_higher,
    "250 + linear": common_results_250_linear,
}

quantile_overall = pd.DataFrame([
    {
        "model": model,
        "sample": "Same 1,140 common targets",
        **summarize_var_results(frame),
    }
    for model, frame in quantile_models.items()
])

quantile_gap_groups = pd.DataFrame([
    {
        "model": model,
        "gap_group": gap_group,
        **summarize_var_results(
            frame.loc[frame["gap_group"].eq(gap_group)]
        ),
    }
    for model, frame in quantile_models.items()
    for gap_group in ["gap_1", "gap_gt1"]
])

for model in quantile_models:
    overall = quantile_overall.loc[
        quantile_overall["model"].eq(model)
    ].iloc[0]

    groups = quantile_gap_groups.loc[
        quantile_gap_groups["model"].eq(model)
    ]

    for column in [
        "evaluated_targets",
        "exceptions",
        "mathematical_ties",
        "near_tie_flags",
    ]:
        assert groups[column].sum() == overall[column]

display(quantile_overall.round(display_rounding))
display(quantile_gap_groups.round(display_rounding))

## 5.6 Inspect paired quantile effects

Linear VaR is lower on all 1,140 common targets, averaging about PHP24,058.72 below higher. All 15 higher-rule exceptions remain exceptions, and two additional targets become exceptions. Fewer exceptions alone do not establish superior calibration.

In [ ]:
# Compare paired exact thresholds before converting differences to display amounts.
# The same target cannot change its underlying loss merely because the quantile rule changes.

pair_columns = [
    "target_interval_id",
    "target_start",
    "target_end",
    "gap_group",
    "var_exact",
    "target_loss_exact",
    "is_exception",
]

paired_quantiles = common_results_250_higher[
    pair_columns
].merge(
    common_results_250_linear[pair_columns],
    on="target_interval_id",
    suffixes=("_higher", "_linear"),
    how="inner",
    validate="one_to_one",
)

assert len(paired_quantiles) == 1140

for column in [
    "target_start",
    "target_end",
    "gap_group",
    "target_loss_exact",
]:
    assert paired_quantiles[f"{column}_higher"].eq(
        paired_quantiles[f"{column}_linear"]
    ).all()

differences_exact = [
    linear - higher
    for higher, linear in zip(
        paired_quantiles["var_exact_higher"],
        paired_quantiles["var_exact_linear"],
    )
]

# Same scenarios: linear cannot exceed higher.
assert all(difference <= 0 for difference in differences_exact)

# A higher-method exception must remain an exception under linear.
assert not (
    paired_quantiles["is_exception_higher"]
    & ~paired_quantiles["is_exception_linear"]
).any()

quantile_estimate_comparison = pd.DataFrame({
    "paired_targets": [len(paired_quantiles)],
    "linear_var_lower": [
        sum(d < 0 for d in differences_exact)
    ],
    "same_var": [
        sum(d == 0 for d in differences_exact)
    ],
    "mean_linear_minus_higher_php": [
        float(
            sum(differences_exact, Fraction(0))
            / len(differences_exact)
        )
    ],
})

paired_quantiles["exception_pattern"] = [
    "Both exceptions" if higher and linear else
    "Linear only" if linear else
    "Higher only" if higher else
    "Neither"
    for higher, linear in zip(
        paired_quantiles["is_exception_higher"],
        paired_quantiles["is_exception_linear"],
    )
]

quantile_switches = (
    paired_quantiles["exception_pattern"]
    .value_counts()
    .reindex(
        ["Neither", "Both exceptions", "Linear only", "Higher only"],
        fill_value=0,
    )
    .rename_axis("exception_pattern")
    .reset_index(name="targets")
)

assert quantile_switches["targets"].sum() == 1140
assert quantile_switches.loc[
    quantile_switches["exception_pattern"].eq("Higher only"),
    "targets",
].iloc[0] == 0

display(
    quantile_estimate_comparison.round({
        "mean_linear_minus_higher_php": 2
    })
)
display(quantile_switches)

## 6. Review-interval inclusion sensitivity

For this separate diagnostic, include review changes in prior training while keeping each main target unchanged. Preserve the main eligibility flags and original interval identities. This isolates the training-pool effect.

In [ ]:
# Include review changes in prior training for this sensitivity only.
# Keep the main target unchanged and use the most recent permitted original intervals without bridging gaps.

def evaluate_inclusion_training(ledger, main_window, method="higher"):
    """Include review changes in training; keep the original main target."""
    ordered = ledger.sort_values("interval_order").copy()
    lookup = ordered.set_index("interval_order", drop=False)

    target_row = lookup.loc[
        int(main_window["target_interval_order"])
    ]

    window_size = int(main_window["window_changes"])

    assert target_row["interval_id"] == main_window["target_interval_id"]
    assert bool(target_row["main_calendar_eligible"])

    # Include all permitted original changes available before this target.
    available_history = ordered.loc[
        ordered["inclusion_calendar_eligible"]
        & ordered["interval_order"].lt(target_row["interval_order"])
        & ordered["end_observation_order"].le(
            target_row["start_observation_order"]
        )
        & ordered["end_source_date"].le(
            target_row["start_source_date"]
        )
    ]

    training = available_history.tail(window_size)

    assert len(training) == window_size
    assert training["interval_id"].is_unique
    assert target_row["interval_id"] not in set(training["interval_id"])

    returns_exact = [
        Fraction(int(numerator), int(denominator))
        for numerator, denominator in zip(
            training["return_numerator"],
            training["return_denominator"],
        )
    ]

    # Estimate before reading the target ending rate.
    prior_var = calculate_var99_exact(
        returns_exact,
        target_row["start_rate_raw"],
        method=method,
    )

    start_rate = Fraction(Decimal(target_row["start_rate_raw"]))
    end_rate = Fraction(Decimal(target_row["end_rate_raw"]))
    target_loss = 1_000_000 * (start_rate - end_rate)

    return {
        "target_interval_id": target_row["interval_id"],
        "target_start": target_row["start_source_date"],
        "target_end": target_row["end_source_date"],
        "gap_group": target_row["gap_group"],
        "window_changes": window_size,
        "method": method,
        "var_exact": prior_var,
        "target_loss_exact": target_loss,
        "target_pnl_exact": -target_loss,
        "review_changes_in_training": int(
            (~training["main_calendar_eligible"]).sum()
        ),
        "training_interval_orders_json": json.dumps(
            training["interval_order"].astype(int).tolist()
        ),
        **classify_loss_exact(target_loss, prior_var),
    }

## 6.1 Evaluate the same 1,390 targets

Training inclusion retains 22 exceptions on the shared main targets. This comparison holds the outcome sample fixed; later additions of targets must be reported separately.

In [ ]:
# Evaluate training inclusion on the identical 1,390 main targets.
# Holding outcomes fixed isolates the effect of changing the historical training pool.

main_ids = set(
    main_results_250_higher["target_interval_id"]
)

sensitivity_windows = (
    main_window_plan.loc[
        main_window_plan["window_changes"].eq(250)
        & main_window_plan["target_interval_id"].isin(main_ids)
    ]
    .sort_values("target_interval_order")
)

assert len(sensitivity_windows) == 1390

inclusion_training_records = [
    evaluate_inclusion_training(
        interval_ledger,
        window,
        method="higher",
    )
    for _, window in sensitivity_windows.iterrows()
]

training_inclusion_250_higher = pd.DataFrame(
    inclusion_training_records
)

assert len(training_inclusion_250_higher) == 1390
assert training_inclusion_250_higher["target_interval_id"].is_unique
assert set(
    training_inclusion_250_higher["target_interval_id"]
) == main_ids

training_sensitivity_models = {
    "Main: review changes excluded": main_results_250_higher,
    "Sensitivity: review changes included": training_inclusion_250_higher,
}

training_sensitivity_overall = pd.DataFrame([
    {
        "training_pool": label,
        **summarize_var_results(frame),
    }
    for label, frame in training_sensitivity_models.items()
])

training_sensitivity_gaps = pd.DataFrame([
    {
        "training_pool": label,
        "gap_group": gap_group,
        **summarize_var_results(
            frame.loc[frame["gap_group"].eq(gap_group)]
        ),
    }
    for label, frame in training_sensitivity_models.items()
    for gap_group in ["gap_1", "gap_gt1"]
])

for label in training_sensitivity_models:
    overall = training_sensitivity_overall.loc[
        training_sensitivity_overall["training_pool"].eq(label)
    ].iloc[0]

    groups = training_sensitivity_gaps.loc[
        training_sensitivity_gaps["training_pool"].eq(label)
    ]

    for column in [
        "evaluated_targets",
        "exceptions",
        "mathematical_ties",
        "near_tie_flags",
    ]:
        assert groups[column].sum() == overall[column]

display(training_sensitivity_overall.round(display_rounding))
display(training_sensitivity_gaps.round(display_rounding))

## 6.2 Inspect changed estimates

Of 1,390 estimates, 18 decrease and 1,372 remain equal; classifications do not switch. Seven hundred and three training windows contain review changes. Limited numerical impact does not resolve uncertain source-calendar evidence.

In [ ]:
# Distinguish a numerical change in VaR from a change in exception classification.
# Unchanged classifications do not resolve the original source-calendar uncertainty.

columns = [
    "target_interval_id",
    "target_start",
    "target_end",
    "gap_group",
    "var_exact",
    "target_loss_exact",
    "is_exception",
]

paired_training_sensitivity = main_results_250_higher[
    columns
].merge(
    training_inclusion_250_higher[columns],
    on="target_interval_id",
    suffixes=("_main", "_included"),
    validate="one_to_one",
    how="inner",
)

assert len(paired_training_sensitivity) == 1390

for column in [
    "target_start",
    "target_end",
    "gap_group",
    "target_loss_exact",
]:
    assert paired_training_sensitivity[f"{column}_main"].eq(
        paired_training_sensitivity[f"{column}_included"]
    ).all()

differences = [
    included - main
    for main, included in zip(
        paired_training_sensitivity["var_exact_main"],
        paired_training_sensitivity["var_exact_included"],
    )
]

training_effect_summary = pd.DataFrame({
    "paired_targets": [len(differences)],
    "included_var_higher": [sum(d > 0 for d in differences)],
    "included_var_lower": [sum(d < 0 for d in differences)],
    "same_var": [sum(d == 0 for d in differences)],
    "mean_included_minus_main_php": [
        float(sum(differences, Fraction(0)) / len(differences))
    ],
    "windows_containing_review_changes": [
        int(
            training_inclusion_250_higher[
                "review_changes_in_training"
            ].gt(0).sum()
        )
    ],
})

paired_training_sensitivity["exception_pattern"] = [
    "Both exceptions" if main and included else
    "Main only" if main else
    "Included-training only" if included else
    "Neither"
    for main, included in zip(
        paired_training_sensitivity["is_exception_main"],
        paired_training_sensitivity["is_exception_included"],
    )
]

training_exception_switches = (
    paired_training_sensitivity["exception_pattern"]
    .value_counts()
    .reindex(
        [
            "Neither",
            "Both exceptions",
            "Main only",
            "Included-training only",
        ],
        fill_value=0,
    )
    .rename_axis("exception_pattern")
    .reset_index(name="targets")
)

assert training_exception_switches["targets"].sum() == 1390

display(training_effect_summary.round({
    "mean_included_minus_main_php": 2
}))
display(training_exception_switches)

## 6.3 Score review targets separately

Three early review targets lack 250 prior inclusion changes and remain unscored. Four later review targets can be scored and are non-exceptions. Four outcomes are too few to validate a separate coverage claim.

In [ ]:
# Score review targets separately so they cannot silently enter the main backtest.
# Insufficient history means not scored, rather than a zero estimate or a non-exception.

# Select the seven original review intervals without changing their flags.
review_targets = (
    interval_ledger.loc[
        ~interval_ledger["main_calendar_eligible"]
    ]
    .sort_values("interval_order")
    .copy()
)

assert len(review_targets) == 7

assert set(
    review_targets[
        ["start_source_row", "end_source_row"]
    ].itertuples(index=False, name=None)
) == frozen_review_pairs

review_records = []

for _, review_target in review_targets.iterrows():

    # Inclusion regime: all permitted original changes BEFORE this target.
    available_history = (
        interval_ledger.loc[
            interval_ledger["inclusion_calendar_eligible"]
            & interval_ledger["interval_order"].lt(
                review_target["interval_order"]
            )
            & interval_ledger["end_observation_order"].le(
                review_target["start_observation_order"]
            )
            & interval_ledger["end_source_date"].le(
                review_target["start_source_date"]
            )
        ]
        .sort_values("interval_order")
    )

    record = {
        "target_interval_id": review_target["interval_id"],
        "target_start": review_target["start_source_date"],
        "target_end": review_target["end_source_date"],
        "gap_calendar_days": review_target["gap_calendar_days"],
        "review_dates_json": review_target["review_dates_json"],
        "available_prior_changes": len(available_history),
        "required_changes": 250,
        "target_status": "insufficient_history",
        "classification": "not_scored",
        "prior_var99_php": float("nan"),
        "target_loss_php": float("nan"),
        "loss_minus_var_php": float("nan"),
        "is_exception": None,
        "is_mathematical_tie": None,
        "near_tie_flag": None,
        "var_exact": None,
        "target_loss_exact": None,
        "training_interval_orders_json": None,
    }

    # Do not invent a VaR estimate when warm-up is incomplete.
    if len(available_history) < 250:
        review_records.append(record)
        continue

    training = available_history.tail(250)

    assert len(training) == 250
    assert training["interval_id"].is_unique
    assert review_target["interval_id"] not in set(
        training["interval_id"]
    )
    assert training["inclusion_calendar_eligible"].all()

    returns_exact = [
        Fraction(int(numerator), int(denominator))
        for numerator, denominator in zip(
            training["return_numerator"],
            training["return_denominator"],
        )
    ]

    # Estimate first, without passing the target ending rate.
    review_var = calculate_var99_exact(
        returns_exact,
        review_target["start_rate_raw"],
        method="higher",
    )

    # Then score the subsequent hypothetical valuation loss.
    start_rate = Fraction(
        Decimal(review_target["start_rate_raw"])
    )
    end_rate = Fraction(
        Decimal(review_target["end_rate_raw"])
    )

    review_loss = 1_000_000 * (start_rate - end_rate)
    classification = classify_loss_exact(review_loss, review_var)

    record.update({
        "target_status": "scored_review",
        "prior_var99_php": float(review_var),
        "target_loss_php": float(review_loss),
        "loss_minus_var_php": float(
            classification["loss_minus_var_exact"]
        ),
        "var_exact": review_var,
        "target_loss_exact": review_loss,
        "training_interval_orders_json": json.dumps(
            training["interval_order"].astype(int).tolist()
        ),
        **classification,
    })

    review_records.append(record)

review_target_results = pd.DataFrame(review_records)

assert len(review_target_results) == 7
assert review_target_results["target_interval_id"].is_unique

display(
    review_target_results[
        [
            "target_start",
            "target_end",
            "gap_calendar_days",
            "available_prior_changes",
            "target_status",
            "prior_var99_php",
            "target_loss_php",
            "loss_minus_var_php",
            "classification",
            "near_tie_flag",
        ]
    ].round({
        "prior_var99_php": 2,
        "target_loss_php": 2,
        "loss_minus_var_php": 2,
    })
)

display(
    review_target_results.groupby("target_status")
    .size()
    .reset_index(name="review_targets")
)

# Summarize only scored review targets—not all seven.
scored_reviews = review_target_results.loc[
    review_target_results["target_status"].eq("scored_review")
].copy()

if len(scored_reviews):
    review_exception_count = int(
        scored_reviews["is_exception"].sum()
    )

    display(pd.DataFrame({
        "sample": ["Review targets only — inclusion regime"],
        "scored_targets": [len(scored_reviews)],
        "exceptions": [review_exception_count],
        "exception_rate_pct": [
            review_exception_count / len(scored_reviews) * 100
        ],
        "mathematical_ties": [
            int(scored_reviews["is_mathematical_tie"].sum())
        ],
    }).round({"exception_rate_pct": 4}))

## 6.4 Reconcile the full inclusion sample

Three earlier ordinary targets and four scoreable review targets extend the shared sample to 1,397 targets, still with 22 exceptions (1.5748%). The lower aggregate rate results from seven added non-exceptions, not improved classifications on shared targets.

In [ ]:
# Reconcile shared targets, earlier ordinary targets and review targets into a separate inclusion sample.
# A lower aggregate rate can arise solely from adding non-exceptions to the denominator.

# 1. Establish the complete inclusion target set.
inclusion_pool = (
    interval_ledger.loc[
        interval_ledger["inclusion_calendar_eligible"]
    ]
    .sort_values("interval_order")
    .reset_index(drop=True)
)

assert len(inclusion_pool) == 1647

# First 250 original changes are warm-up; subsequent rows are targets.
full_inclusion_targets = inclusion_pool.iloc[250:].copy()

assert len(full_inclusion_targets) == 1397

full_target_ids = set(full_inclusion_targets["interval_id"])

shared_ids = set(
    training_inclusion_250_higher["target_interval_id"]
)
assert len(shared_ids) == 1390
assert shared_ids.issubset(full_target_ids)

# Separate newly available targets into review and ordinary intervals.
added_targets = full_inclusion_targets.loc[
    ~full_inclusion_targets["interval_id"].isin(shared_ids)
].copy()

added_reviews = added_targets.loc[
    ~added_targets["main_calendar_eligible"]
]
earlier_ordinary_targets = added_targets.loc[
    added_targets["main_calendar_eligible"]
]

assert len(added_targets) == 7
assert len(added_reviews) == 4
assert len(earlier_ordinary_targets) == 3

assert earlier_ordinary_targets["end_source_date"].lt(
    main_results_250_higher["target_end"].min()
).all()

# Confirm that the four review outcomes were already scored.
scored_review_rows = review_target_results.loc[
    review_target_results["target_status"].eq("scored_review")
].copy()

assert set(
    scored_review_rows["target_interval_id"]
) == set(added_reviews["interval_id"])


# 2. Score the three newly available ordinary targets.
earlier_records = []

for _, row in earlier_ordinary_targets.iterrows():
    target_reference = {
        "target_interval_order": int(row["interval_order"]),
        "target_interval_id": row["interval_id"],
        "window_changes": 250,
    }

    # These are main calendar candidates, so the existing helper applies.
    result = evaluate_inclusion_training(
        interval_ledger,
        target_reference,
        method="higher",
    )

    earlier_records.append(result)

earlier_ordinary_results = pd.DataFrame(earlier_records)


# 3. Normalize and combine the three result sources.
core_columns = [
    "target_interval_id",
    "target_start",
    "target_end",
    "var_exact",
    "target_loss_exact",
    "loss_minus_var_exact",
    "classification",
    "is_exception",
    "is_mathematical_tie",
    "near_tie_flag",
    "training_interval_orders_json",
]

shared_part = training_inclusion_250_higher[core_columns].copy()
shared_part["target_origin"] = "Shared main target"

review_part = scored_review_rows[core_columns].copy()
review_part["target_origin"] = "Review target"

earlier_part = earlier_ordinary_results[core_columns].copy()
earlier_part["target_origin"] = "Earlier ordinary target"

full_inclusion_results = pd.concat(
    [shared_part, review_part, earlier_part],
    ignore_index=True,
)

# Attach original identity and reporting-group metadata.
target_metadata = interval_ledger[
    ["interval_id", "interval_order", "gap_group"]
].rename(columns={
    "interval_id": "target_interval_id",
    "interval_order": "target_interval_order",
})

full_inclusion_results = (
    full_inclusion_results
    .merge(
        target_metadata,
        on="target_interval_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values("target_interval_order")
    .reset_index(drop=True)
)

full_inclusion_results["window_changes"] = 250
full_inclusion_results["method"] = "higher"
full_inclusion_results["eligibility_regime"] = "full_inclusion"

assert len(full_inclusion_results) == 1397
assert full_inclusion_results["target_interval_id"].is_unique
assert set(
    full_inclusion_results["target_interval_id"]
) == full_target_ids

# Every retained result must have a resolved classification.
for column in [
    "is_exception",
    "is_mathematical_tie",
    "near_tie_flag",
]:
    assert full_inclusion_results[column].notna().all()
    full_inclusion_results[column] = (
        full_inclusion_results[column].astype(bool)
    )

# Verify saved membership against the last 250 inclusion changes.
for row in full_inclusion_results.itertuples(index=False):
    expected_training = inclusion_pool.loc[
        inclusion_pool["interval_order"].lt(
            row.target_interval_order
        )
    ].tail(250)

    actual_keys = json.loads(row.training_interval_orders_json)

    assert actual_keys == (
        expected_training["interval_order"].astype(int).tolist()
    )
    assert len(actual_keys) == 250
    assert row.target_interval_order not in actual_keys
    assert expected_training["end_source_date"].le(
        row.target_start
    ).all()

# Numeric convenience columns; exact results remain unchanged.
for exact_column, numeric_column in {
    "var_exact": "prior_var99_php",
    "target_loss_exact": "target_loss_php",
    "loss_minus_var_exact": "loss_minus_var_php",
}.items():
    full_inclusion_results[numeric_column] = (
        full_inclusion_results[exact_column].map(float)
    )


# 4. Inspect the three earlier ordinary targets.
display(
    full_inclusion_results.loc[
        full_inclusion_results["target_origin"].eq(
            "Earlier ordinary target"
        ),
        [
            "target_start",
            "target_end",
            "gap_group",
            "prior_var99_php",
            "target_loss_php",
            "classification",
        ],
    ].round({
        "prior_var99_php": 2,
        "target_loss_php": 2,
    })
)

# Reconcile contributions from each source.
origin_summary = pd.DataFrame([
    {
        "target_origin": origin,
        **summarize_var_results(group),
    }
    for origin, group in full_inclusion_results.groupby(
        "target_origin"
    )
])

full_inclusion_summary = pd.DataFrame([{
    "sample": "Full inclusion — 250 + higher",
    "first_target_start": (
        full_inclusion_results["target_start"].min()
    ),
    "last_target_end": (
        full_inclusion_results["target_end"].max()
    ),
    **summarize_var_results(full_inclusion_results),
}])

assert origin_summary["evaluated_targets"].sum() == 1397
assert origin_summary["exceptions"].sum() == int(
    full_inclusion_results["is_exception"].sum()
)

display(origin_summary.round(display_rounding))
display(full_inclusion_summary.round(display_rounding))

## 7. Frozen evidence and source-audit status

The original 27-file validation bundle remains the baseline evidence. Verify its stored hashes read-only. The reader edition adds explanations and supplementary coverage analysis; it must not overwrite the frozen code snapshot.

The bounded source audit covers all 22 main exceptions / 44 unique endpoints. [Remaining gap=1 audit](../reports/source_audit_remaining_gap1_v1/SOURCE_AUDIT_SUMMARY.md) and the [gap-crossing audit](../reports/source_audit_gap_crossing_v1/SOURCE_AUDIT_SUMMARY.md) retain original PDFs, rates and lineage. One mislinked November 2021 attachment was rejected and resolved through the correct official archive. Source matching does not certify historical release times or every training value.

In [ ]:
# Verify the frozen evidence read-only after this notebook's presentation changes.
# Checksums establish unchanged files; they do not establish model adequacy or publication timing.

# Verify the existing baseline without re-exporting this reader edition.
validation_bundle_path = project_root / "reports/var_validation_v1"
validation_manifest = json.loads(
    (validation_bundle_path / "checksums.json").read_text()
)
for relative_name, expected_sha256 in validation_manifest.items():
    actual_sha256 = hashlib.sha256(
        (validation_bundle_path / relative_name).read_bytes()
    ).hexdigest()
    assert actual_sha256 == expected_sha256, relative_name

print(f"Verified {len(validation_manifest)} frozen validation files; none replaced.")
print("Validation files:", validation_bundle_path)


## 8. Coverage and uncertainty

Summarize the observed exception frequency and a 95% Wilson binomial reference interval. Here 99% describes the VaR quantile; 95% describes the uncertainty procedure for an exception proportion. Independence and a constant exception probability are not established, so these are supplementary reference intervals, not automatic acceptance tests.

Method reference: [NIST Wilson interval](https://www.itl.nist.gov/div898/handbook/prc/section2/prc241.htm). The ideal rank reference assumes continuous, exchangeable observations and is unconditional; it is not a guaranteed probability for each forecast or gap group.

In [ ]:
# Use the frozen classifications rather than recomputing exceptions from rounded displayed amounts.
# The Wilson interval is a binomial reference that assumes independent trials with a common probability.

from pathlib import Path
import hashlib
import json
from math import sqrt
from statistics import NormalDist

import pandas as pd

# Resolve the project independently so this section also works in a fresh kernel.
project_candidates = [Path.cwd(), *Path.cwd().parents,
                      Path(".")]
project_root = next(
    candidate for candidate in project_candidates
    if (candidate / "reports/var_validation_v1/checksums.json").is_file()
)

# Read the frozen main-model results.
bundle = project_root / "reports" / "var_validation_v1"
relative_path = "results/main_250_higher_full.csv"
results_path = bundle / relative_path

checksums = json.loads((bundle / "checksums.json").read_text())
assert hashlib.sha256(results_path.read_bytes()).hexdigest() == (
    checksums[relative_path]
)

coverage_results = pd.read_csv(results_path)

# Use the saved exact-arithmetic classifications.
assert coverage_results["classification"].isin(
    ["exception", "non_exception", "mathematical_tie"]
).all()

coverage_results["exception_flag"] = (
    coverage_results["classification"] == "exception"
)

assert len(coverage_results) == 1390
assert coverage_results["exception_flag"].sum() == 22
assert coverage_results["target_interval_id"].is_unique
assert set(coverage_results["gap_group"]) == {"gap_1", "gap_gt1"}


def wilson_interval(exceptions, targets, confidence=0.95):
    """Binomial reference interval; assumes independent, common-rate trials."""
    z = NormalDist().inv_cdf(1 - (1 - confidence) / 2)
    proportion = exceptions / targets
    denominator = 1 + z**2 / targets

    center = (
        proportion + z**2 / (2 * targets)
    ) / denominator

    half_width = (
        z * sqrt(
            proportion * (1 - proportion) / targets
            + z**2 / (4 * targets**2)
        )
    ) / denominator

    return center - half_width, center + half_width


groups = [
    ("Overall", coverage_results),
    (
        "gap_1",
        coverage_results.loc[coverage_results["gap_group"] == "gap_1"],
    ),
    (
        "gap_gt1",
        coverage_results.loc[coverage_results["gap_group"] == "gap_gt1"],
    ),
]

summary_rows = []

for label, group in groups:
    targets = len(group)
    exceptions = int(group["exception_flag"].sum())
    lower, upper = wilson_interval(exceptions, targets)

    summary_rows.append({
        "sample": label,
        "evaluated_targets": targets,
        "exceptions": exceptions,
        "exception_rate_pct": 100 * exceptions / targets,
        "nominal_1pct_reference_count": targets * 0.01,
        "wilson_95_lower_pct": 100 * lower,
        "wilson_95_upper_pct": 100 * upper,
    })

coverage_summary = pd.DataFrame(summary_rows)

display(coverage_summary.round(4))

# Higher quantile: rank 248 of 250 = third-largest scenario loss.
# Under ideal continuous, exchangeable observations:
# next-observation exceedance probability = 3 / 251.
rank_reference_probability = 3 / 251

display(pd.DataFrame({
    "reference": [
        "Nominal 99% VaR reference",
        "250/higher ideal rank reference",
    ],
    "exception_probability_pct": [
        1.0,
        100 * rank_reference_probability,
    ],
    "reference_count_for_1390_targets": [
        1390 * 0.01,
        1390 * rank_reference_probability,
    ],
}).round(4))

print(
    "Wilson intervals are supplementary binomial reference intervals. "
    "Independence and a constant exception probability are not established. "
    "The rank reference is an idealized unconditional reference, "
    "not a guaranteed rate for either gap group."
)

### Coverage interpretation

The overall exception rate is **1.5827%**, with a 95% Wilson reference interval of **1.0475%–2.3848%**. The one-calendar-day interval is **0.6433%–1.9518%**, while the multi-calendar-day interval is **1.7008%–5.6386%**. Under the reference assumptions, the nominal 1% lies inside the first gap group's interval but outside the overall and multi-calendar-day intervals. The broad multi-day interval also reflects its smaller sample.

For 1,390 targets, nominal 1% corresponds to 13.9 exceptions. The rank-248 `higher` cutoff has an ideal continuous/exchangeable unconditional reference of **3/251 ≈ 1.1952%**, or 16.61 exceptions. Observed exceptions exceed both references. These comparisons signal a concern, not a definitive verdict under actual mixed-duration and potentially changing conditions.

## 9. Yearly exception counts and rates

Assign the year using the **target ending source date**. Keep target counts beside rates: one exception among 52 targets is about 1.92%, whereas two among 186 is about 1.08%. Annual rates based on few exceptions can vary substantially. 2021 begins after warm-up and 2026 ends in September; excluded review targets remain excluded.

In [ ]:
# Assign each interval to its ending source year and retain separate gap-group denominators.
# Partial-year coverage and small counts limit comparisons; yearly rates are descriptive, not causal evidence.

# Use the same frozen main-model results from the previous cell.
yearly_results = coverage_results.copy()

yearly_results["target_end"] = pd.to_datetime(
    yearly_results["target_end"],
    errors="raise",
)

yearly_results["year"] = yearly_results["target_end"].dt.year

# Count targets and exceptions separately for each year and gap group.
yearly_gap_summary = (
    yearly_results
    .groupby(["year", "gap_group"], as_index=False)
    .agg(
        evaluated_targets=("target_interval_id", "size"),
        exceptions=("exception_flag", "sum"),
    )
    .sort_values(["year", "gap_group"])
    .reset_index(drop=True)
)

yearly_gap_summary["exceptions"] = (
    yearly_gap_summary["exceptions"].astype(int)
)

yearly_gap_summary["exception_rate_pct"] = (
    100
    * yearly_gap_summary["exceptions"]
    / yearly_gap_summary["evaluated_targets"]
)

yearly_gap_summary["nominal_1pct_reference_count"] = (
    yearly_gap_summary["evaluated_targets"] * 0.01
)

yearly_gap_summary["coverage_note"] = (
    yearly_gap_summary["year"]
    .map({
        2021: "Evaluated targets begin January 20",
        2026: "Partial year: through September 30",
    })
    .fillna("Within full calendar-year coverage")
)

# Reconcile yearly totals with the original full-sample results.
assert yearly_gap_summary["evaluated_targets"].sum() == 1390
assert yearly_gap_summary["exceptions"].sum() == 22

for gap_group in ["gap_1", "gap_gt1"]:
    original_group = yearly_results.loc[
        yearly_results["gap_group"] == gap_group
    ]

    summarized_group = yearly_gap_summary.loc[
        yearly_gap_summary["gap_group"] == gap_group
    ]

    assert summarized_group["evaluated_targets"].sum() == len(
        original_group
    )
    assert summarized_group["exceptions"].sum() == int(
        original_group["exception_flag"].sum()
    )

display(yearly_gap_summary.round({
    "exception_rate_pct": 4,
    "nominal_1pct_reference_count": 2,
}))

# Compact comparison tables.
print("Exception counts:")
display(
    yearly_gap_summary.pivot(
        index="year",
        columns="gap_group",
        values="exceptions",
    )
)

print("Exception rates (%):")
display(
    yearly_gap_summary.pivot(
        index="year",
        columns="gap_group",
        values="exception_rate_pct",
    ).round(4)
)

print("Passed: yearly counts reconcile to 1,390 targets and 22 exceptions.")

### Yearly interpretation

**Seven of ten multi-calendar-day exceptions occur in 2021–2022:** 7/110 targets (6.36%), compared with 3/211 (1.42%) in 2023–September 2026. The larger contribution to the aggregate multi-day concern is concentrated in the first two evaluated years. This is a retrospective description, not a new holdout split or evidence of permanent improvement.

One-calendar-day annual rates range from 0.53% to 1.66%. Multi-calendar-day rates exceed the one-calendar-day rate in five of six reported years, but counts are small and this is not a formal difference test. Zero exceptions in 60 multi-day targets in 2025 means no observed threshold exceedance in that subset, not zero loss or zero future risk.

## Validation conclusion

### Overall assessment

The historical-simulation VaR implementation is reproducible and
traceable, but its coverage results require qualification. The model
provides an interpretable research baseline for a fixed, unhedged
USD1 million position valued in PHP. It is not a validated operational
bank risk model.

### Supported strengths

- **Reproducible results.** A fresh project kernel reproduced all
  27 checksummed validation files. Exact values, settings and training
  memberships are preserved.
- **Checked calculations.** Sign, unit, quantile, tie-handling and
  outcome-isolation checks passed. Exceptions use strict comparison
  of exact signed loss against exact signed VaR.
- **Traceable exception inputs.** All 44 unique endpoints associated
  with the 22 main exceptions match the archived workbook and the
  corresponding BSP bulletin USD-row PHIL. PESO EQUIVALENT values.
  Original interval identities and valuation losses reconcile.
- **Transparent sensitivity analysis.** Alternative window and
  quantile settings were compared on the same 1,140 targets.
  Review-interval inclusion was assessed separately.
- **Separate interval reporting.** One-calendar-day and
  multi-calendar-day targets are reported separately, making an
  important difference in observed coverage visible.

These strengths support implementation integrity. They do not, by
themselves, establish forecast adequacy.

### Coverage concerns

The main 250-change, higher-quantile model produced **22 exceptions
across 1,390 targets: 1.5827%**.

| Sample | Targets | Exceptions | Exception rate |
|---|---:|---:|---:|
| Overall | 1,390 | 22 | 1.5827% |
| One-calendar-day intervals | 1,069 | 12 | 1.1225% |
| Multi-calendar-day intervals | 321 | 10 | 3.1153% |

The overall rate exceeds the nominal 1% reference. Multi-calendar-day
intervals show a substantially higher observed exception frequency
than one-calendar-day intervals.

Seven of the ten multi-calendar-day exceptions occurred in 2021–2022:
7/110 targets, compared with 3/211 in 2023–September 2026.
This concentration is descriptive evidence, not proof of a causal
calendar-gap effect or sustained later improvement.

Loss severity also matters. The largest main-sample loss was
**PHP878,000**, exceeding its prior VaR of approximately
**PHP425,719.97** by **PHP452,280.03**. An exception rate alone does
not describe the size of losses beyond VaR.

### Sensitivity findings

On the same 1,140 targets:

| Configuration | Exceptions | Exception rate |
|---|---:|---:|
| 250 + higher | 15 | 1.3158% |
| 500 + higher | 18 | 1.5789% |
| 250 + linear | 17 | 1.4912% |

A longer window did not consistently produce higher VaR estimates
or fewer exceptions. Linear interpolation produced lower estimates
than higher on all common targets and added two exceptions.

Including the seven review changes in training did not change
exception classifications on the shared 1,390 targets, although
18 VaR estimates decreased. This supports limited classification
sensitivity to that specific inclusion choice; it does not resolve
the underlying source uncertainty.

No alternative is declared universally superior from these results.
The agreed main specification remains unchanged.

### Uncertainty and clustering

Supplementary 95% Wilson binomial reference intervals are:

| Sample | Lower bound | Upper bound |
|---|---:|---:|
| Overall | 1.0475% | 2.3848% |
| One-calendar-day intervals | 0.6433% | 1.9518% |
| Multi-calendar-day intervals | 1.7008% | 5.6386% |

These intervals assume independent trials with a common exception
probability. Those conditions are not established for this rolling,
mixed-duration sample. The intervals therefore provide qualified
reference information rather than a definitive acceptance decision.

The nominal 1% reference corresponds to 13.9 exceptions over
1,390 targets. Under ideal continuous, exchangeable observations,
the rank-248 cutoff used by 250/higher has an unconditional
exceedance reference of 3/251, approximately 1.1952%, corresponding
to 16.61 exceptions. This idealized reference is not a guaranteed
conditional probability for the actual forecasts or either gap group.

No consecutive exceptions were observed among 1,385 valid adjacent
evaluated pairs. However, absence of consecutive exceptions does not
establish independence. The exploratory random-placement diagnostic
found a three-exception span as close as the observed span in 11.615%
of placements. It did not provide strong evidence of unusual
closeness under that particular benchmark; it does not prove that
clustering or dependence is absent.

### Limitations

- Source-date gaps do not establish identical economic measurement
  horizons. Results describe published-observation intervals, not
  verified market-close one-day trading risk.
- Historical release availability, revisions and full-period
  construction continuity remain incompletely verified. Sequence
  checks prevent future-target inclusion but do not establish what
  was publicly available at each historical forecast origin.
- Source auditing covers exception endpoints, not every observation
  or training input.
- Only a few observations determine the 99% tail estimate in a
  250-change window. Tail estimates are sensitive to sample
  composition and market conditions.
- The study uses one fixed, unhedged USD position and excludes
  changing holdings, hedges, transaction costs and other portfolio
  risk factors.
- Diagnostics and sensitivity comparisons are retrospective.
  No untouched prospective evaluation or regulatory acceptance is
  established.
- VaR is a quantile estimate, not a maximum-loss guarantee. It does
  not describe all possible extreme losses.

### Implication for stress testing

Retain historical VaR as the study's transparent baseline and report
its coverage concerns explicitly. Supplement it with predefined
stress scenarios to show financial impacts under specified adverse
movements.

Stress scenarios complement VaR; they do not repair its coverage or
prove forecast accuracy. The BPI Group disclosure-based illustration
will remain a separate supporting case with its own exposure date,
assumptions and limitations.